**Public archival copy.** Saved cell outputs (including embedded images) and Colab-specific runtime metadata have been removed. Original code and explanatory cell content remain unchanged. Some code refers to Google Drive paths and intermediate research artifacts that are not distributed here. The Fitzpatrick17k image files and the researchers' individual access link are **not included**. See `notebooks/README.md` for access and reproducibility notes.


# Phase 6 — Domain Adaptation / Mitigation

## Domain-Invariant and Explainable Deep Learning for Equitable and Generalizable Skin Lesion Malignancy Classification Across Diverse Skin Tones

### Objective

Phase 6 evaluates whether explicit domain adaptation can reduce Fair–Dark domain disparity without materially sacrificing discrimination or malignant-lesion performance.

The primary proposed method is **class-conditional Maximum Mean Discrepancy (MMD) regularization** applied to the ResNet50 avgpool representation.

This notebook uses the **frozen Corrected Phase 4 and Corrected Phase 5 branches** as authoritative upstream sources.

### Core design principles

- Do not alter the fixed Fair/Dark test sets.
- Do not overwrite Corrected Phase 4 or Corrected Phase 5 artifacts.
- Use validation data only for hyperparameter selection.
- Keep the seed-specific trained model as the independent experimental unit.
- Do not use test data for bandwidth selection, lambda selection, early stopping, or model selection.
- Preserve class balance inside domain-adaptation batches.
- Separate exploratory tuning from final confirmatory evaluation.

In [ ]:
# ============================================
# Section 1.1
# Mount Google Drive
# ============================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

In [ ]:
# ============================================
# Section 1.2
# Imports
# ============================================

from pathlib import Path
import os
import json
import random
import math
import time
import hashlib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import (
    models,
    transforms
)

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("PyTorch:", torch.__version__)

In [ ]:
# ============================================
# Section 1.3
# Reproducibility and Device
# ============================================

GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================
# Section 1.4
# Phase Paths
# ============================================

ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)

DATA_PATH = (
    ARTIFACT_PATH
    / "dataset_splits"
)

CORRECTED_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments_Corrected"
)

CORRECTED_PHASE5_PATH = (
    ARTIFACT_PATH
    / "Phase_5_Mechanistic_Analysis_Corrected"
)

PHASE6_PATH = (
    ARTIFACT_PATH
    / "Phase_6_Domain_Adaptation"
)

PHASE6_CONFIG_PATH = PHASE6_PATH / "config"
PHASE6_MODEL_PATH = PHASE6_PATH / "models"
PHASE6_HISTORY_PATH = PHASE6_PATH / "histories"
PHASE6_PREDICTION_PATH = PHASE6_PATH / "predictions"
PHASE6_TABLE_PATH = PHASE6_PATH / "tables"
PHASE6_FIGURE_PATH = PHASE6_PATH / "figures"
PHASE6_MANIFEST_PATH = PHASE6_PATH / "manifests"
PHASE6_AUDIT_PATH = PHASE6_PATH / "audits"
PHASE6_TUNING_PATH = PHASE6_PATH / "tuning"
PHASE6_FINAL_PATH = PHASE6_PATH / "final_models"

for directory in [
    PHASE6_PATH,
    PHASE6_CONFIG_PATH,
    PHASE6_MODEL_PATH,
    PHASE6_HISTORY_PATH,
    PHASE6_PREDICTION_PATH,
    PHASE6_TABLE_PATH,
    PHASE6_FIGURE_PATH,
    PHASE6_MANIFEST_PATH,
    PHASE6_AUDIT_PATH,
    PHASE6_TUNING_PATH,
    PHASE6_FINAL_PATH
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Phase 6 root:", PHASE6_PATH)

# Section 2 — Frozen Upstream Audit

Phase 6 must begin only if both corrected upstream phases are complete and frozen.

Required upstream sources:

- Corrected Phase 4 completion manifest
- Corrected Phase 4 repeated-run result table
- Corrected Phase 4 model checkpoints
- Corrected Phase 5 completion manifest

The historical/original Phase 4 and Phase 5 branches are not used as training inputs for Phase 6.

In [ ]:
# ============================================
# Section 2.1
# Required Upstream Files
# ============================================

PHASE4_COMPLETION_FILE = (
    CORRECTED_PHASE4_PATH
    / "manifests"
    / "Phase4_Corrected_Completion_Manifest.json"
)

PHASE4_RESULTS_FILE = (
    CORRECTED_PHASE4_PATH
    / "tables"
    / "Phase4_Corrected_Repeated_Run_Results.csv"
)

PHASE4_MODEL_PATH = (
    CORRECTED_PHASE4_PATH
    / "models"
)

PHASE5_COMPLETION_FILE = (
    CORRECTED_PHASE5_PATH
    / "manifests"
    / "Phase5_Corrected_Completion_Manifest.json"
)

required_upstream_files = [
    PHASE4_COMPLETION_FILE,
    PHASE4_RESULTS_FILE,
    PHASE5_COMPLETION_FILE
]

for file_path in required_upstream_files:
    print(
        file_path,
        "->",
        file_path.exists()
    )

In [ ]:
# ============================================
# Section 2.2
# Read Frozen Completion Manifests
# ============================================

with open(
    PHASE4_COMPLETION_FILE,
    "r"
) as f:
    phase4_manifest = json.load(f)

with open(
    PHASE5_COMPLETION_FILE,
    "r"
) as f:
    phase5_manifest = json.load(f)

print(
    "Corrected Phase 4 status:",
    phase4_manifest.get("Status")
)

print(
    "Corrected Phase 5 status:",
    phase5_manifest.get("Status")
)

print(
    "Corrected Phase 5 frozen:",
    phase5_manifest.get("Frozen")
)

In [ ]:
# ============================================
# Section 2.3A
# Inspect Upstream Manifest Status Fields
# ============================================

print(
    "Phase 4 manifest status:",
    phase4_manifest.get("Status")
)

print(
    "Phase 5 manifest status:",
    phase5_manifest.get("Status")
)

print(
    "Phase 5 frozen:",
    phase5_manifest.get("Frozen")
)


print(
    "\nPhase 4 manifest:"
)

print(
    json.dumps(
        phase4_manifest,
        indent=4
    )
)

In [ ]:
# ============================================
# Section 2.3
# Hard Upstream Gate
# Corrected Version
# ============================================

phase4_status = str(
    phase4_manifest.get(
        "status",
        ""
    )
).strip().upper()


phase5_status = str(
    phase5_manifest.get(
        "Status",
        ""
    )
).strip().upper()


assert (
    phase4_status
    ==
    "COMPLETE"
), (
    "Corrected Phase 4 is not marked COMPLETE. "
    f"Found: {phase4_manifest.get('status')}"
)


assert (
    phase5_status
    ==
    "COMPLETE"
), (
    "Corrected Phase 5 is not marked COMPLETE. "
    f"Found: {phase5_manifest.get('Status')}"
)


assert (
    phase4_manifest.get(
        "checkpoint_audit_passed"
    )
    is True
), (
    "Corrected Phase 4 checkpoint audit did not pass."
)


assert (
    phase4_manifest.get(
        "metric_audit_passed"
    )
    is True
), (
    "Corrected Phase 4 metric audit did not pass."
)


assert (
    phase4_manifest.get(
        "train_validation_test_overlap"
    )
    ==
    0
), (
    "Corrected Phase 4 has train/validation/test overlap."
)


assert (
    phase5_manifest.get(
        "Frozen"
    )
    is True
), (
    "Corrected Phase 5 is not frozen."
)


print(
    "✓ Corrected Phase 4 status: COMPLETE"
)

print(
    "✓ Phase 4 checkpoint audit: PASSED"
)

print(
    "✓ Phase 4 metric audit: PASSED"
)

print(
    "✓ Phase 4 train/evaluation overlap: 0"
)

print(
    "✓ Corrected Phase 5 status: COMPLETE"
)

print(
    "✓ Corrected Phase 5 frozen: True"
)

print(
    "\n✓ Upstream corrected branches verified."
)

# Section 3 — Phase 6 Experimental Design

## Starting point

The primary Phase 6 adaptation experiment will use the **Corrected M2 (50% Fair / 50% Dark)** composition as the controlled training base.

This isolates the effect of explicit feature-distribution alignment from simple changes in training-domain proportion.

## Proposed class-conditional adaptation loss

For each mini-batch:

$$
L_{\text{total}}
=
L_{\text{CE}}
+
\frac{\lambda}{2}
\left(
\operatorname{MMD}_B^2
+
\operatorname{MMD}_M^2
\right)
$$

where:

- $L_{\text{CE}}$ is classification cross-entropy,
- $\operatorname{MMD}_B^2$ aligns Fair and Dark benign representations,
- $\operatorname{MMD}_M^2$ aligns Fair and Dark malignant representations,
- $\lambda$ controls alignment strength.

### Balanced adaptation mini-batch

Target composition per batch:

- 8 Fair benign
- 8 Fair malignant
- 8 Dark benign
- 8 Dark malignant

Total batch size:

$$
32
$$

### Hyperparameter-selection stage

Initial lambda grid:

$$
\lambda\in\{0,1,10,50,100\}
$$

Development seeds:

$$
42,43,44
$$

Only validation metrics will be used to select lambda.

Final test evaluation will occur only after the adaptation configuration is frozen.

In [ ]:
# ============================================
# Section 3.1
# Phase 6 Configuration
# ============================================

BASE_EXPERIMENT = "M2_50F_50D"
BASELINE_TRAIN_SIZE = 800

BATCH_COMPOSITION = {
    "Fair_Benign": 8,
    "Fair_Malignant": 8,
    "Dark_Benign": 8,
    "Dark_Malignant": 8
}

ADAPTATION_BATCH_SIZE = sum(
    BATCH_COMPOSITION.values()
)

LAMBDA_GRID = [
    0.0,
    1.0,
    10.0,
    50.0,
    100.0
]

TUNING_SEEDS = [
    42,
    43,
    44
]

FINAL_SEEDS = list(
    range(
        42,
        52
    )
)

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10

NUM_CLASSES = 2
FEATURE_DIMENSION = 2048

print("Base experiment:", BASE_EXPERIMENT)
print("Adaptation batch size:", ADAPTATION_BATCH_SIZE)
print("Lambda grid:", LAMBDA_GRID)
print("Tuning seeds:", TUNING_SEEDS)
print("Final seeds:", FINAL_SEEDS)

In [ ]:
# ============================================
# Section 3.2
# Save Initial Phase 6 Configuration
# ============================================

phase6_config = {

    "Phase":
        "Phase 6",

    "Title":
        "Domain Adaptation / Mitigation",

    "Authoritative_Upstream":
        [
            "Corrected Phase 4",
            "Corrected Phase 5"
        ],

    "Base_Experiment":
        BASE_EXPERIMENT,

    "Method":
        "Class-Conditional MMD Regularization",

    "Feature_Layer":
        "ResNet50 avgpool",

    "Feature_Dimension":
        FEATURE_DIMENSION,

    "Batch_Composition":
        BATCH_COMPOSITION,

    "Batch_Size":
        ADAPTATION_BATCH_SIZE,

    "Lambda_Grid":
        LAMBDA_GRID,

    "Tuning_Seeds":
        TUNING_SEEDS,

    "Final_Seeds":
        FINAL_SEEDS,

    "Learning_Rate":
        LEARNING_RATE,

    "Weight_Decay":
        WEIGHT_DECAY,

    "Max_Epochs":
        MAX_EPOCHS,

    "Early_Stopping_Patience":
        EARLY_STOPPING_PATIENCE,

    "Test_Data_Used_For_Tuning":
        False,

    "Status":
        "SETUP"
}

PHASE6_CONFIG_FILE = (
    PHASE6_CONFIG_PATH
    / "Phase6_Initial_Configuration.json"
)

with open(
    PHASE6_CONFIG_FILE,
    "w"
) as f:
    json.dump(
        phase6_config,
        f,
        indent=4
    )

print("✓ Configuration saved:")
print(PHASE6_CONFIG_FILE)

# Section 4 — Dataset Discovery and Audit

The next step is to identify the exact corrected M2 training and validation CSV files and verify:

- expected row counts,
- exact Fair/Dark composition,
- exact benign/malignant composition,
- no duplicate images,
- no overlap with fixed Fair/Dark test sets.

No adaptation training should begin until these audits pass.

In [ ]:
# ============================================
# Section 4.1
# Inspect Available Corrected Phase 4
# Training and Validation Files
# ============================================

PHASE4_TRAINING_SET_PATH = (
    CORRECTED_PHASE4_PATH
    / "training_sets"
)

PHASE4_VALIDATION_SET_PATH = (
    CORRECTED_PHASE4_PATH
    / "validation_sets"
)

print("Training-set files:")

for path in sorted(
    PHASE4_TRAINING_SET_PATH.glob("*.csv")
):
    print(path.name)

print("\nValidation-set files:")

for path in sorted(
    PHASE4_VALIDATION_SET_PATH.glob("*.csv")
):
    print(path.name)

In [ ]:
# ============================================
# Section 4.2
# Inspect Fixed Evaluation Splits
# ============================================

fair_test_file = DATA_PATH / "fair_test.csv"
dark_test_file = DATA_PATH / "dark_test.csv"

print("Fair test exists:", fair_test_file.exists())
print("Dark test exists:", dark_test_file.exists())

if (
    fair_test_file.exists()
    and
    dark_test_file.exists()
):

    fair_test_df = pd.read_csv(
        fair_test_file
    )

    dark_test_df = pd.read_csv(
        dark_test_file
    )

    print(
        "\nFair test:",
        fair_test_df.shape
    )

    print(
        fair_test_df[
            "target"
        ].value_counts()
        .sort_index()
    )

    print(
        "\nDark test:",
        dark_test_df.shape
    )

    print(
        dark_test_df[
            "target"
        ].value_counts()
        .sort_index()
    )

In [ ]:
# ============================================
# Section 4.3
# Audit Corrected M2 Training Set
# ============================================

M2_TRAIN_FILE = (

    PHASE4_TRAINING_SET_PATH
    / "M2_50F_50D_train.csv"
)


m2_train_df = pd.read_csv(
    M2_TRAIN_FILE
)


print(
    "Shape:",
    m2_train_df.shape
)


print(
    "\nColumns:"
)

print(
    list(
        m2_train_df.columns
    )
)


print(
    "\nTarget distribution:"
)

print(
    m2_train_df[
        "target"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nSkin-tone distribution:"
)

print(
    m2_train_df[
        "skin_tone"
    ]
    .value_counts()
)


print(
    "\nDomain × Class:"
)

print(
    pd.crosstab(

        m2_train_df[
            "skin_tone"
        ],

        m2_train_df[
            "target"
        ]
    )
)


print(
    "\nDuplicate MD5 hashes:",
    m2_train_df[
        "md5hash"
    ]
    .duplicated()
    .sum()
)


print(
    "\nMissing images:",
    (
        ~m2_train_df[
            "image_path"
        ]
        .map(
            lambda x: Path(x).exists()
        )
    )
    .sum()
)

In [ ]:
# ============================================
# Section 4.4
# Locate Existing Validation Artifacts
# ============================================

print(
    "Searching Corrected Phase 4..."
)


corrected_validation_candidates = []


for path in (
    CORRECTED_PHASE4_PATH
    .rglob("*")
):

    if (
        path.is_file()
        and
        (
            "val" in path.name.lower()
            or
            "validation" in path.name.lower()
        )
    ):

        corrected_validation_candidates.append(
            path
        )


for path in (
    corrected_validation_candidates
):

    print(
        path
    )


print(
    "\nTotal corrected candidates:",
    len(
        corrected_validation_candidates
    )
)

In [ ]:
# ============================================
# Section 4.5
# Locate Original Phase 4 Validation Files
# ============================================

ORIGINAL_PHASE4_PATH = (

    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments"
)


original_validation_candidates = []


if ORIGINAL_PHASE4_PATH.exists():

    for path in (
        ORIGINAL_PHASE4_PATH
        .rglob("*")
    ):

        if (
            path.is_file()
            and
            (
                "val" in path.name.lower()
                or
                "validation" in path.name.lower()
            )
        ):

            original_validation_candidates.append(
                path
            )


for path in (
    original_validation_candidates
):

    print(
        path
    )


print(
    "\nTotal original candidates:",
    len(
        original_validation_candidates
    )
)

In [ ]:
# ============================================
# Section 4.6
# Load Exact M2 Validation Set
# Used by Corrected Phase 4
# ============================================

M2_VAL_FILE = (

    ORIGINAL_PHASE4_PATH
    / "validation_sets"
    / "M2_50F_50D_val.csv"
)


assert (
    M2_VAL_FILE.exists()
), (
    f"M2 validation file not found: "
    f"{M2_VAL_FILE}"
)


m2_val_df = pd.read_csv(
    M2_VAL_FILE
)


print(
    "Validation file:"
)

print(
    M2_VAL_FILE
)


print(
    "\nShape:",
    m2_val_df.shape
)


print(
    "\nColumns:"
)

print(
    list(
        m2_val_df.columns
    )
)


print(
    "\nTarget distribution:"
)

print(
    m2_val_df[
        "target"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nSkin-tone distribution:"
)

print(
    m2_val_df[
        "skin_tone"
    ]
    .value_counts()
)


print(
    "\nDomain × Class:"
)

print(
    pd.crosstab(

        m2_val_df[
            "skin_tone"
        ],

        m2_val_df[
            "target"
        ]
    )
)


print(
    "\nDuplicate MD5 hashes:",
    m2_val_df[
        "md5hash"
    ]
    .duplicated()
    .sum()
)


print(
    "\nMissing images:",
    (
        ~m2_val_df[
            "image_path"
        ]
        .map(
            lambda x: Path(x).exists()
        )
    )
    .sum()
)

In [ ]:
# ============================================
# Section 4.7
# Train / Validation / Test Leakage Audit
# ============================================

train_hashes = set(
    m2_train_df[
        "md5hash"
    ]
    .astype(str)
)


val_hashes = set(
    m2_val_df[
        "md5hash"
    ]
    .astype(str)
)


fair_test_hashes = set(
    fair_test_df[
        "md5hash"
    ]
    .astype(str)
)


dark_test_hashes = set(
    dark_test_df[
        "md5hash"
    ]
    .astype(str)
)


test_hashes = (
    fair_test_hashes
    |
    dark_test_hashes
)


train_val_overlap = (
    train_hashes
    &
    val_hashes
)


train_test_overlap = (
    train_hashes
    &
    test_hashes
)


val_test_overlap = (
    val_hashes
    &
    test_hashes
)


fair_dark_test_overlap = (
    fair_test_hashes
    &
    dark_test_hashes
)


print(
    "Train ↔ Validation overlap:",
    len(
        train_val_overlap
    )
)


print(
    "Train ↔ Test overlap:",
    len(
        train_test_overlap
    )
)


print(
    "Validation ↔ Test overlap:",
    len(
        val_test_overlap
    )
)


print(
    "Fair-test ↔ Dark-test overlap:",
    len(
        fair_dark_test_overlap
    )
)


assert (
    len(
        train_val_overlap
    )
    ==
    0
), (
    "Train-validation leakage detected."
)


assert (
    len(
        train_test_overlap
    )
    ==
    0
), (
    "Train-test leakage detected."
)


assert (
    len(
        val_test_overlap
    )
    ==
    0
), (
    "Validation-test leakage detected."
)


assert (
    len(
        fair_dark_test_overlap
    )
    ==
    0
), (
    "Fair/Dark test overlap detected."
)


print(
    "\n✓ No MD5 leakage detected."
)

In [ ]:
# ============================================
# Section 4.8
# Hard Dataset Composition Gate
# ============================================

# --------------------------------------------
# Training
# --------------------------------------------

assert (
    len(
        m2_train_df
    )
    ==
    800
)


assert (
    (
        m2_train_df[
            "target"
        ]
        ==
        0
    )
    .sum()
    ==
    400
)


assert (
    (
        m2_train_df[
            "target"
        ]
        ==
        1
    )
    .sum()
    ==
    400
)


assert (
    (
        m2_train_df[
            "skin_tone"
        ]
        ==
        "Fair"
    )
    .sum()
    ==
    400
)


assert (
    (
        m2_train_df[
            "skin_tone"
        ]
        ==
        "Dark"
    )
    .sum()
    ==
    400
)


for domain_name in [
    "Fair",
    "Dark"
]:

    for target_value in [
        0,
        1
    ]:

        count = (

            (
                (
                    m2_train_df[
                        "skin_tone"
                    ]
                    ==
                    domain_name
                )

                &

                (
                    m2_train_df[
                        "target"
                    ]
                    ==
                    target_value
                )
            )
            .sum()
        )


        assert (
            count
            ==
            200
        ), (
            f"Unexpected M2 train count: "
            f"{domain_name}, class {target_value}, "
            f"found {count}"
        )


# --------------------------------------------
# Validation
# --------------------------------------------

assert (
    len(
        m2_val_df
    )
    ==
    200
)


for domain_name in [
    "Fair",
    "Dark"
]:

    for target_value in [
        0,
        1
    ]:

        count = (

            (
                (
                    m2_val_df[
                        "skin_tone"
                    ]
                    ==
                    domain_name
                )

                &

                (
                    m2_val_df[
                        "target"
                    ]
                    ==
                    target_value
                )
            )
            .sum()
        )


        assert (
            count
            ==
            50
        ), (
            f"Unexpected M2 validation count: "
            f"{domain_name}, class {target_value}, "
            f"found {count}"
        )


print(
    "✓ M2 training composition verified."
)

print(
    "✓ M2 validation composition verified."
)

print(
    "✓ Dataset composition gate passed."
)

In [ ]:
# ============================================
# Section 4.9
# Freeze Phase 6 Dataset Sources
# ============================================

phase6_dataset_manifest = {

    "Base_Experiment":
        "M2_50F_50D",

    "Training_File":
        str(
            M2_TRAIN_FILE
        ),

    "Validation_File":
        str(
            M2_VAL_FILE
        ),

    "Fair_Test_File":
        str(
            fair_test_file
        ),

    "Dark_Test_File":
        str(
            dark_test_file
        ),

    "Training_Size":
        int(
            len(
                m2_train_df
            )
        ),

    "Validation_Size":
        int(
            len(
                m2_val_df
            )
        ),

    "Fair_Test_Size":
        int(
            len(
                fair_test_df
            )
        ),

    "Dark_Test_Size":
        int(
            len(
                dark_test_df
            )
        ),

    "Training_Composition": {

        "Fair_Benign":
            200,

        "Fair_Malignant":
            200,

        "Dark_Benign":
            200,

        "Dark_Malignant":
            200
    },

    "Validation_Composition": {

        "Fair_Benign":
            50,

        "Fair_Malignant":
            50,

        "Dark_Benign":
            50,

        "Dark_Malignant":
            50
    },

    "Train_Validation_Overlap":
        len(
            train_val_overlap
        ),

    "Train_Test_Overlap":
        len(
            train_test_overlap
        ),

    "Validation_Test_Overlap":
        len(
            val_test_overlap
        ),

    "Frozen":
        True
}


PHASE6_DATA_MANIFEST_FILE = (

    PHASE6_MANIFEST_PATH
    / "Phase6_Dataset_Manifest.json"
)


with open(
    PHASE6_DATA_MANIFEST_FILE,
    "w"
) as f:

    json.dump(

        phase6_dataset_manifest,

        f,

        indent=4
    )


print(
    "✓ Phase 6 dataset manifest saved."
)

print(
    PHASE6_DATA_MANIFEST_FILE
)

In [ ]:
# ============================================
# Section 4.10
# Recover Exact Phase 4 Training Transform
# ============================================

from pathlib import Path
import json


SEARCH_ROOT = Path(
    "/content/drive/MyDrive"
)


candidate_notebooks = []


for notebook_path in SEARCH_ROOT.rglob(
    "*.ipynb"
):

    name_lower = (
        notebook_path.name
        .lower()
    )


    if (
        "phase_4" in name_lower
        or
        "phase4" in name_lower
        or
        "controlled_mixing" in name_lower
    ):

        candidate_notebooks.append(
            notebook_path
        )


print(
    "Candidate Phase 4 notebooks:"
)


for path in candidate_notebooks:

    print(
        path
    )


print(
    "\nSearching notebook cells..."
)


search_terms = [

    "train_transform",
    "RandomHorizontalFlip",
    "RandomRotation",
    "ColorJitter",
    "RandomResizedCrop",
    "Resize"
]


matches = []


for notebook_path in candidate_notebooks:

    try:

        with open(
            notebook_path,
            "r",
            encoding="utf-8"
        ) as f:

            notebook_json = json.load(
                f
            )


        for cell_number, cell in enumerate(
            notebook_json.get(
                "cells",
                []
            )
        ):

            source = "".join(
                cell.get(
                    "source",
                    []
                )
            )


            if any(
                term in source
                for term in search_terms
            ):

                matches.append({

                    "Notebook":
                        str(
                            notebook_path
                        ),

                    "Cell":
                        cell_number,

                    "Source":
                        source
                })


    except Exception as error:

        print(
            "Could not inspect:",
            notebook_path,
            error
        )


print(
    "\nMatches found:",
    len(
        matches
    )
)


for match in matches:

    print(
        "\n" + "=" * 100
    )

    print(
        "Notebook:",
        match[
            "Notebook"
        ]
    )

    print(
        "Cell:",
        match[
            "Cell"
        ]
    )

    print(
        "-" * 100
    )

    print(
        match[
            "Source"
        ]
    )

In [ ]:
# ============================================
# Section 5.0
# Restore Frozen Preprocessing Configuration
# ============================================

PREPROCESSING_CONFIG_FILE = (
    ARTIFACT_PATH
    / "preprocessing_config.json"
)


assert (
    PREPROCESSING_CONFIG_FILE.exists()
), (
    f"Preprocessing config not found: "
    f"{PREPROCESSING_CONFIG_FILE}"
)


with open(
    PREPROCESSING_CONFIG_FILE,
    "r"
) as f:

    preprocessing_config = json.load(
        f
    )


IMAGE_SIZE = int(
    preprocessing_config[
        "image_size"
    ]
)


NORMALIZE_MEAN = (
    preprocessing_config[
        "normalization"
    ][
        "mean"
    ]
)


NORMALIZE_STD = (
    preprocessing_config[
        "normalization"
    ][
        "std"
    ]
)


print(
    "Image size:",
    IMAGE_SIZE
)


print(
    "Normalization mean:",
    NORMALIZE_MEAN
)


print(
    "Normalization std:",
    NORMALIZE_STD
)

In [ ]:
# ============================================
# Section 5.1
# Freeze Phase 6 Train / Evaluation Transforms
# ============================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])


print(
    "✓ Phase 6 train transform frozen."
)

print(
    "✓ Phase 6 evaluation transform frozen."
)

In [ ]:
# ============================================
# Section 5.2
# Phase 6 Dataset Class
# ============================================

class Phase6Dataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        idx
    ):

        row = (
            self.dataframe
            .iloc[idx]
        )


        image = Image.open(
            row[
                "image_path"
            ]
        ).convert(
            "RGB"
        )


        if self.transform is not None:

            image = self.transform(
                image
            )


        target = int(
            row[
                "target"
            ]
        )


        domain = (
            0
            if
            row[
                "skin_tone"
            ]
            ==
            "Fair"
            else
            1
        )


        return {

            "image":
                image,

            "target":
                torch.tensor(
                    target,
                    dtype=torch.long
                ),

            "domain":
                torch.tensor(
                    domain,
                    dtype=torch.long
                ),

            "index":
                int(
                    idx
                ),

            "md5hash":
                row[
                    "md5hash"
                ]
        }

In [ ]:
# ============================================
# Section 5.3
# Balanced 8/8/8/8 Batch Sampler
# ============================================

from torch.utils.data import Sampler


class BalancedDomainClassBatchSampler(
    Sampler
):

    def __init__(
        self,
        dataframe,
        samples_per_group=8,
        seed=42
    ):

        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.samples_per_group = (
            samples_per_group
        )

        self.seed = seed

        self.epoch = 0


        self.group_indices = {}


        group_definitions = {

            "Fair_Benign":
                (
                    "Fair",
                    0
                ),

            "Fair_Malignant":
                (
                    "Fair",
                    1
                ),

            "Dark_Benign":
                (
                    "Dark",
                    0
                ),

            "Dark_Malignant":
                (
                    "Dark",
                    1
                )
        }


        for group_name, (
            domain_name,
            target_value

        ) in group_definitions.items():

            indices = (

                self.dataframe.index[

                    (
                        self.dataframe[
                            "skin_tone"
                        ]
                        ==
                        domain_name
                    )

                    &

                    (
                        self.dataframe[
                            "target"
                        ]
                        ==
                        target_value
                    )
                ]

                .to_numpy()
            )


            self.group_indices[
                group_name
            ] = indices


        group_sizes = [

            len(
                values
            )

            for values
            in self.group_indices.values()
        ]


        self.num_batches = min(

            size
            //
            self.samples_per_group

            for size in group_sizes
        )


    def set_epoch(
        self,
        epoch
    ):

        self.epoch = epoch


    def __len__(
        self
    ):

        return self.num_batches


    def __iter__(
        self
    ):

        rng = np.random.default_rng(

            self.seed
            +
            self.epoch
        )


        shuffled_groups = {}


        for group_name, indices in (
            self.group_indices.items()
        ):

            shuffled = (
                indices.copy()
            )

            rng.shuffle(
                shuffled
            )

            shuffled_groups[
                group_name
            ] = shuffled


        for batch_idx in range(
            self.num_batches
        ):

            batch_indices = []


            start = (

                batch_idx
                *
                self.samples_per_group
            )

            end = (

                start
                +
                self.samples_per_group
            )


            for group_name in [

                "Fair_Benign",
                "Fair_Malignant",
                "Dark_Benign",
                "Dark_Malignant"

            ]:

                batch_indices.extend(

                    shuffled_groups[
                        group_name
                    ][
                        start:end
                    ]
                    .tolist()
                )


            rng.shuffle(
                batch_indices
            )


            yield batch_indices

In [ ]:
# ============================================
# Section 5.4
# Build Balanced Adaptation DataLoader
# ============================================

phase6_train_dataset = Phase6Dataset(

    dataframe=m2_train_df,

    transform=train_transform
)


balanced_batch_sampler = (
    BalancedDomainClassBatchSampler(

        dataframe=m2_train_df,

        samples_per_group=8,

        seed=GLOBAL_SEED
    )
)


phase6_train_loader = DataLoader(

    phase6_train_dataset,

    batch_sampler=balanced_batch_sampler,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Training images:",
    len(
        phase6_train_dataset
    )
)


print(
    "Balanced batches per epoch:",
    len(
        phase6_train_loader
    )
)


print(
    "Images per balanced batch:",
    32
)

In [ ]:
# ============================================
# Section 5.5
# Audit Balanced Batch Composition
# ============================================

balanced_batch_sampler.set_epoch(
    0
)


batch_audit_rows = []


for batch_number, batch in enumerate(
    phase6_train_loader
):

    targets = (
        batch[
            "target"
        ]
        .numpy()
    )

    domains = (
        batch[
            "domain"
        ]
        .numpy()
    )


    counts = {

        "Fair_Benign":
            int(
                (
                    (domains == 0)
                    &
                    (targets == 0)
                )
                .sum()
            ),

        "Fair_Malignant":
            int(
                (
                    (domains == 0)
                    &
                    (targets == 1)
                )
                .sum()
            ),

        "Dark_Benign":
            int(
                (
                    (domains == 1)
                    &
                    (targets == 0)
                )
                .sum()
            ),

        "Dark_Malignant":
            int(
                (
                    (domains == 1)
                    &
                    (targets == 1)
                )
                .sum()
            )
    }


    batch_audit_rows.append({

        "Batch":
            batch_number,

        **counts
    })


balanced_batch_audit_df = (
    pd.DataFrame(
        batch_audit_rows
    )
)


display(
    balanced_batch_audit_df
)


print(
    "\nUnique compositions:"
)

print(
    balanced_batch_audit_df[

        [
            "Fair_Benign",
            "Fair_Malignant",
            "Dark_Benign",
            "Dark_Malignant"
        ]

    ]
    .drop_duplicates()
)


assert (
    balanced_batch_audit_df[
        [
            "Fair_Benign",
            "Fair_Malignant",
            "Dark_Benign",
            "Dark_Malignant"
        ]
    ]
    .eq(8)
    .all()
    .all()
)


print(
    "\n✓ Every batch is exactly 8/8/8/8."
)

In [ ]:
# ============================================
# Section 6.1
# Deterministic Training Dataset
# for MMD Bandwidth Estimation
# ============================================

phase6_bandwidth_dataset = Phase6Dataset(

    dataframe=m2_train_df,

    transform=eval_transform
)


phase6_bandwidth_loader = DataLoader(

    phase6_bandwidth_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Bandwidth dataset size:",
    len(
        phase6_bandwidth_dataset
    )
)

In [ ]:
# ============================================
# Section 6.2
# ResNet50 Classifier Returning Avgpool Features
# ============================================

from torchvision.models import (
    ResNet50_Weights
)


class ResNet50WithFeatures(
    nn.Module
):

    def __init__(
        self,
        pretrained=True
    ):

        super().__init__()


        if pretrained:

            weights = (
                ResNet50_Weights
                .IMAGENET1K_V1
            )

        else:

            weights = None


        backbone = models.resnet50(
            weights=weights
        )


        self.feature_extractor = nn.Sequential(

            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,

            backbone.avgpool
        )


        self.classifier = nn.Linear(
            2048,
            2
        )


    def forward(
        self,
        x
    ):

        features = (
            self.feature_extractor(
                x
            )
        )


        features = torch.flatten(
            features,
            1
        )


        logits = self.classifier(
            features
        )


        return (
            logits,
            features
        )

In [ ]:
# ============================================
# Section 6.3
# Model Output Sanity Check
# ============================================

test_phase6_model = (
    ResNet50WithFeatures(
        pretrained=True
    )
    .to(
        DEVICE
    )
)


sample_batch = next(
    iter(
        phase6_bandwidth_loader
    )
)


sample_images = (
    sample_batch[
        "image"
    ]
    .to(
        DEVICE
    )
)


with torch.no_grad():

    sample_logits, sample_features = (
        test_phase6_model(
            sample_images
        )
    )


print(
    "Image batch:",
    sample_images.shape
)

print(
    "Logits:",
    sample_logits.shape
)

print(
    "Features:",
    sample_features.shape
)


assert (
    sample_logits.shape[1]
    ==
    2
)


assert (
    sample_features.shape[1]
    ==
    2048
)


print(
    "\n✓ Phase 6 model architecture verified."
)

In [ ]:
# ============================================
# Section 6.4
# Extract Pre-Adaptation Training Features
# ============================================

def extract_training_features(
    model,
    dataloader
):

    model.eval()


    feature_batches = []
    target_batches = []
    domain_batches = []
    index_batches = []


    with torch.no_grad():

        for batch in dataloader:

            images = (
                batch[
                    "image"
                ]
                .to(
                    DEVICE
                )
            )


            _, features = model(
                images
            )


            feature_batches.append(

                features
                .detach()
                .cpu()
                .numpy()
            )


            target_batches.append(

                batch[
                    "target"
                ]
                .numpy()
            )


            domain_batches.append(

                batch[
                    "domain"
                ]
                .numpy()
            )


            index_batches.append(

                np.asarray(
                    batch[
                        "index"
                    ]
                )
            )


    return (

        np.concatenate(
            feature_batches,
            axis=0
        ),

        np.concatenate(
            target_batches
        ),

        np.concatenate(
            domain_batches
        ),

        np.concatenate(
            index_batches
        )
    )


(
    initial_train_features,
    initial_train_targets,
    initial_train_domains,
    initial_train_indices

) = extract_training_features(

    test_phase6_model,
    phase6_bandwidth_loader
)


print(
    "Features:",
    initial_train_features.shape
)

print(
    "Targets:",
    initial_train_targets.shape
)

print(
    "Domains:",
    initial_train_domains.shape
)


print(
    "Finite:",
    np.isfinite(
        initial_train_features
    )
    .all()
)

In [ ]:
# ============================================
# Section 6.5
# Median-Heuristic Bandwidth
# ============================================

def pairwise_squared_distances_numpy(
    x
):

    x = np.asarray(
        x,
        dtype=np.float64
    )


    norms = np.sum(
        x ** 2,
        axis=1,
        keepdims=True
    )


    distances = (

        norms
        +
        norms.T
        -
        2.0
        *
        np.matmul(
            x,
            x.T
        )
    )


    return np.maximum(
        distances,
        0.0
    )


def median_bandwidth_numpy(
    x
):

    sq_dist = (
        pairwise_squared_distances_numpy(
            x
        )
    )


    upper = (

        sq_dist[
            np.triu_indices(
                len(x),
                k=1
            )
        ]
    )


    upper = (
        upper[
            upper > 0
        ]
    )


    if len(
        upper
    ) == 0:

        raise ValueError(
            "No positive pairwise distances."
        )


    return float(

        np.sqrt(
            np.median(
                upper
            )
        )
    )

In [ ]:
# ============================================
# Section 6.6
# Estimate Training-Only MMD Bandwidths
# ============================================

TRAINING_MMD_BANDWIDTHS = {}


bandwidth_rows = []


for class_name, target_value in [

    (
        "Benign",
        0
    ),

    (
        "Malignant",
        1
    )

]:

    class_features = (

        initial_train_features[

            initial_train_targets
            ==
            target_value
        ]
    )


    sigma = median_bandwidth_numpy(
        class_features
    )


    TRAINING_MMD_BANDWIDTHS[
        class_name
    ] = sigma


    bandwidth_rows.append({

        "Class":
            class_name,

        "N":
            len(
                class_features
            ),

        "Sigma":
            sigma
    })


phase6_bandwidth_df = pd.DataFrame(
    bandwidth_rows
)


display(
    phase6_bandwidth_df
)


print(
    "\nFrozen training-only bandwidths:"
)

print(
    TRAINING_MMD_BANDWIDTHS
)

In [ ]:
# ============================================
# Section 6.7
# Save Training-Only Bandwidth Configuration
# ============================================

PHASE6_BANDWIDTH_FILE = (

    PHASE6_CONFIG_PATH
    / "Phase6_Training_Only_MMD_Bandwidths.csv"
)


phase6_bandwidth_df.to_csv(

    PHASE6_BANDWIDTH_FILE,

    index=False
)


bandwidth_manifest = {

    "Source":
        "Corrected M2 training set only",

    "Training_Images":
        800,

    "Benign_Images":
        400,

    "Malignant_Images":
        400,

    "Representation":
        "ImageNet-initialized ResNet50 avgpool",

    "Feature_Dimension":
        2048,

    "Transform":
        "Deterministic evaluation transform",

    "Benign_Sigma":
        float(
            TRAINING_MMD_BANDWIDTHS[
                "Benign"
            ]
        ),

    "Malignant_Sigma":
        float(
            TRAINING_MMD_BANDWIDTHS[
                "Malignant"
            ]
        ),

    "Validation_Used":
        False,

    "Test_Used":
        False,

    "Frozen":
        True
}


with open(

    PHASE6_CONFIG_PATH
    / "Phase6_MMD_Bandwidth_Manifest.json",

    "w"

) as f:

    json.dump(
        bandwidth_manifest,
        f,
        indent=4
    )


print(
    "✓ Training-only MMD bandwidths saved and frozen."
)

In [ ]:
# ============================================
# Section 7.1
# Differentiable Gaussian Kernel
# ============================================

def gaussian_kernel_torch(
    x,
    y,
    sigma
):

    squared_distance = torch.cdist(
        x,
        y,
        p=2
    ) ** 2


    return torch.exp(

        -squared_distance

        /

        (
            2.0
            *
            sigma ** 2
        )
    )

In [ ]:
# ============================================
# Section 7.2
# Differentiable Biased MMD Squared
# Training Objective
# ============================================

def training_mmd2_torch(
    x,
    y,
    sigma
):

    if (
        x.shape[0] < 1
        or
        y.shape[0] < 1
    ):

        raise ValueError(
            "Both domains must contain samples."
        )


    k_xx = gaussian_kernel_torch(
        x,
        x,
        sigma
    )


    k_yy = gaussian_kernel_torch(
        y,
        y,
        sigma
    )


    k_xy = gaussian_kernel_torch(
        x,
        y,
        sigma
    )


    mmd2 = (

        k_xx.mean()

        +

        k_yy.mean()

        -

        2.0
        *
        k_xy.mean()
    )


    # Numerical safety only.
    # The biased empirical estimator is theoretically >= 0.
    return torch.clamp(
        mmd2,
        min=0.0
    )

In [ ]:
# ============================================
# Section 7.3
# Class-Conditional Fair-Dark MMD Loss
# ============================================

def class_conditional_mmd_loss(
    features,
    targets,
    domains,
    benign_sigma,
    malignant_sigma
):

    fair_mask = (
        domains
        ==
        0
    )

    dark_mask = (
        domains
        ==
        1
    )


    benign_mask = (
        targets
        ==
        0
    )

    malignant_mask = (
        targets
        ==
        1
    )


    fair_benign = features[

        fair_mask
        &
        benign_mask
    ]


    dark_benign = features[

        dark_mask
        &
        benign_mask
    ]


    fair_malignant = features[

        fair_mask
        &
        malignant_mask
    ]


    dark_malignant = features[

        dark_mask
        &
        malignant_mask
    ]


    benign_mmd2 = (
        training_mmd2_torch(

            fair_benign,

            dark_benign,

            benign_sigma
        )
    )


    malignant_mmd2 = (
        training_mmd2_torch(

            fair_malignant,

            dark_malignant,

            malignant_sigma
        )
    )


    total_mmd = (

        benign_mmd2
        +
        malignant_mmd2

    ) / 2.0


    return (
        total_mmd,
        benign_mmd2,
        malignant_mmd2
    )

In [ ]:
# ============================================
# Section 7.4
# MMD Loss Sanity Check
# ============================================

balanced_batch_sampler.set_epoch(
    0
)


sanity_batch = next(
    iter(
        phase6_train_loader
    )
)


sanity_images = (

    sanity_batch[
        "image"
    ]
    .to(
        DEVICE
    )
)


sanity_targets = (

    sanity_batch[
        "target"
    ]
    .to(
        DEVICE
    )
)


sanity_domains = (

    sanity_batch[
        "domain"
    ]
    .to(
        DEVICE
    )
)


test_phase6_model.train()


sanity_logits, sanity_features = (
    test_phase6_model(
        sanity_images
    )
)


sanity_ce = F.cross_entropy(

    sanity_logits,

    sanity_targets
)


(
    sanity_mmd,
    sanity_benign_mmd,
    sanity_malignant_mmd

) = class_conditional_mmd_loss(

    features=sanity_features,

    targets=sanity_targets,

    domains=sanity_domains,

    benign_sigma=(
        TRAINING_MMD_BANDWIDTHS[
            "Benign"
        ]
    ),

    malignant_sigma=(
        TRAINING_MMD_BANDWIDTHS[
            "Malignant"
        ]
    )
)


sanity_total_loss = (

    sanity_ce

    +

    sanity_mmd
)


print(
    "Cross-entropy:",
    float(
        sanity_ce.detach().cpu()
    )
)


print(
    "Benign MMD²:",
    float(
        sanity_benign_mmd.detach().cpu()
    )
)


print(
    "Malignant MMD²:",
    float(
        sanity_malignant_mmd.detach().cpu()
    )
)


print(
    "Mean class-conditional MMD²:",
    float(
        sanity_mmd.detach().cpu()
    )
)


print(
    "Total loss at lambda=1:",
    float(
        sanity_total_loss
        .detach()
        .cpu()
    )
)


assert torch.isfinite(
    sanity_total_loss
)


print(
    "\n✓ Differentiable class-conditional MMD is operational."
)

In [ ]:
# ============================================
# Section 8.1
# Build Separate Fair and Dark
# Validation DataLoaders
# ============================================

fair_val_df = (

    m2_val_df[

        m2_val_df[
            "skin_tone"
        ]
        ==
        "Fair"
    ]

    .reset_index(
        drop=True
    )
)


dark_val_df = (

    m2_val_df[

        m2_val_df[
            "skin_tone"
        ]
        ==
        "Dark"
    ]

    .reset_index(
        drop=True
    )
)


fair_val_dataset = Phase6Dataset(

    dataframe=fair_val_df,

    transform=eval_transform
)


dark_val_dataset = Phase6Dataset(

    dataframe=dark_val_df,

    transform=eval_transform
)


fair_val_loader = DataLoader(

    fair_val_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


dark_val_loader = DataLoader(

    dark_val_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Fair validation:",
    len(
        fair_val_dataset
    )
)


print(
    "Dark validation:",
    len(
        dark_val_dataset
    )
)


print(
    "\nFair class counts:"
)

print(
    fair_val_df[
        "target"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nDark class counts:"
)

print(
    dark_val_df[
        "target"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================
# Section 8.2
# Binary Validation Metrics
# ============================================

def compute_phase6_metrics(
    labels,
    probabilities,
    predictions
):

    labels = np.asarray(
        labels
    )

    probabilities = np.asarray(
        probabilities
    )

    predictions = np.asarray(
        predictions
    )


    accuracy = accuracy_score(
        labels,
        predictions
    )


    precision = precision_score(

        labels,
        predictions,

        zero_division=0
    )


    recall = recall_score(

        labels,
        predictions,

        zero_division=0
    )


    f1 = f1_score(

        labels,
        predictions,

        zero_division=0
    )


    auc = roc_auc_score(
        labels,
        probabilities
    )


    tn, fp, fn, tp = confusion_matrix(

        labels,
        predictions,

        labels=[
            0,
            1
        ]

    ).ravel()


    specificity = (

        tn
        /
        (tn + fp)

        if
        (tn + fp) > 0

        else np.nan
    )


    fnr = (

        fn
        /
        (fn + tp)

        if
        (fn + tp) > 0

        else np.nan
    )


    return {

        "Accuracy":
            float(
                accuracy
            ),

        "Precision":
            float(
                precision
            ),

        "Recall":
            float(
                recall
            ),

        "F1":
            float(
                f1
            ),

        "AUC":
            float(
                auc
            ),

        "Specificity":
            float(
                specificity
            ),

        "FNR":
            float(
                fnr
            ),

        "TN":
            int(
                tn
            ),

        "FP":
            int(
                fp
            ),

        "FN":
            int(
                fn
            ),

        "TP":
            int(
                tp
            )
    }

In [ ]:
# ============================================
# Section 8.3
# Evaluate Model on One Validation Domain
# ============================================

def evaluate_phase6_model(
    model,
    dataloader
):

    model.eval()


    all_labels = []
    all_probabilities = []
    all_predictions = []


    running_loss = 0.0
    total_samples = 0


    with torch.no_grad():

        for batch in dataloader:

            images = (

                batch[
                    "image"
                ]
                .to(
                    DEVICE
                )
            )


            targets = (

                batch[
                    "target"
                ]
                .to(
                    DEVICE
                )
            )


            logits, _ = model(
                images
            )


            loss = F.cross_entropy(

                logits,

                targets
            )


            probabilities = torch.softmax(

                logits,

                dim=1

            )[:, 1]


            predictions = torch.argmax(

                logits,

                dim=1
            )


            batch_size = (
                targets.shape[0]
            )


            running_loss += (

                float(
                    loss.detach().cpu()
                )

                *
                batch_size
            )


            total_samples += (
                batch_size
            )


            all_labels.extend(

                targets
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


            all_probabilities.extend(

                probabilities
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


            all_predictions.extend(

                predictions
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )


    metrics = compute_phase6_metrics(

        labels=all_labels,

        probabilities=all_probabilities,

        predictions=all_predictions
    )


    metrics[
        "Loss"
    ] = (

        running_loss
        /
        total_samples
    )


    return metrics

In [ ]:
# ============================================
# Section 8.4
# Fair + Dark Validation Evaluation
# ============================================

def evaluate_both_validation_domains(
    model
):

    fair_metrics = (
        evaluate_phase6_model(

            model,

            fair_val_loader
        )
    )


    dark_metrics = (
        evaluate_phase6_model(

            model,

            dark_val_loader
        )
    )


    summary = {

        "Fair_AUC":
            fair_metrics[
                "AUC"
            ],

        "Dark_AUC":
            dark_metrics[
                "AUC"
            ],

        "Fair_Recall":
            fair_metrics[
                "Recall"
            ],

        "Dark_Recall":
            dark_metrics[
                "Recall"
            ],

        "Fair_F1":
            fair_metrics[
                "F1"
            ],

        "Dark_F1":
            dark_metrics[
                "F1"
            ],

        "Fair_Loss":
            fair_metrics[
                "Loss"
            ],

        "Dark_Loss":
            dark_metrics[
                "Loss"
            ]
    }


    summary[
        "Worst_Domain_AUC"
    ] = min(

        summary[
            "Fair_AUC"
        ],

        summary[
            "Dark_AUC"
        ]
    )


    summary[
        "Mean_Domain_AUC"
    ] = (

        summary[
            "Fair_AUC"
        ]

        +

        summary[
            "Dark_AUC"
        ]

    ) / 2.0


    summary[
        "AUC_Gap"
    ] = abs(

        summary[
            "Fair_AUC"
        ]

        -

        summary[
            "Dark_AUC"
        ]
    )


    summary[
        "Recall_Gap"
    ] = abs(

        summary[
            "Fair_Recall"
        ]

        -

        summary[
            "Dark_Recall"
        ]
    )


    return summary

In [ ]:
# ============================================
# Section 9.1
# Freeze Validation-Only Selection Rule
# ============================================

WORST_AUC_TOLERANCE = 0.002


PHASE6_SELECTION_RULE = {

    "Primary":
        "Maximize worst-domain validation AUC",

    "Tie_Tolerance":
        WORST_AUC_TOLERANCE,

    "Secondary":
        "Minimize malignant recall disparity",

    "Tertiary":
        "Maximize mean-domain validation AUC",

    "Final_Tie_Break":
        "Prefer smaller lambda",

    "Validation_Only":
        True,

    "Test_Used":
        False,

    "Frozen":
        True
}


with open(

    PHASE6_CONFIG_PATH
    / "Phase6_Model_Selection_Rule.json",

    "w"

) as f:

    json.dump(

        PHASE6_SELECTION_RULE,

        f,

        indent=4
    )


print(
    "✓ Phase 6 validation-selection rule frozen."
)

In [ ]:
# ============================================
# Section 10.1
# Resume-Safe Run Artifact Paths
# ============================================

def lambda_to_name(
    lambda_value
):

    return str(
        lambda_value
    ).replace(
        ".",
        "p"
    )


def phase6_run_directory(
    lambda_value,
    seed
):

    return (

        PHASE6_TUNING_PATH

        / (
            f"lambda_{lambda_to_name(lambda_value)}"
            f"_seed_{seed}"
        )
    )


def phase6_run_paths(
    lambda_value,
    seed
):

    run_dir = (
        phase6_run_directory(

            lambda_value,
            seed
        )
    )


    run_dir.mkdir(

        parents=True,

        exist_ok=True
    )


    return {

        "directory":
            run_dir,

        "last_checkpoint":
            run_dir
            / "last_checkpoint.pth",

        "best_checkpoint":
            run_dir
            / "best_model.pth",

        "history":
            run_dir
            / "history.csv",

        "manifest":
            run_dir
            / "run_manifest.json"
    }

In [ ]:
# ============================================
# Section 10.2
# Run-Specific Reproducibility
# ============================================

def set_run_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

In [ ]:
# ============================================
# Section 10.3
# Create Fresh Adaptation Model
# ============================================

def create_phase6_model():

    model = ResNet50WithFeatures(
        pretrained=True
    )


    return model.to(
        DEVICE
    )

In [ ]:
# ============================================
# Section 10.4
# Run Completion Detection
# ============================================

def tuning_run_complete(
    lambda_value,
    seed
):

    paths = phase6_run_paths(

        lambda_value,
        seed
    )


    if not paths[
        "manifest"
    ].exists():

        return False


    try:

        with open(
            paths[
                "manifest"
            ],
            "r"
        ) as f:

            manifest = json.load(
                f
            )


        return (

            manifest.get(
                "Status"
            )
            ==
            "COMPLETE"

            and

            paths[
                "best_checkpoint"
            ].exists()
        )


    except Exception:

        return False

In [ ]:
# ============================================
# Section 10.5
# Epoch-Level Resume Checkpoint
# ============================================

def save_phase6_checkpoint(
    path,
    model,
    optimizer,
    epoch,
    best_selection_score,
    best_epoch,
    epochs_without_improvement,
    lambda_value,
    seed,
    history
):

    torch.save({

        "epoch":
            int(
                epoch
            ),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "best_selection_score":
            best_selection_score,

        "best_epoch":
            int(
                best_epoch
            ),

        "epochs_without_improvement":
            int(
                epochs_without_improvement
            ),

        "lambda_value":
            float(
                lambda_value
            ),

        "seed":
            int(
                seed
            ),

        "history":
            history

    }, path)

In [ ]:
# ============================================
# Section 10.6
# Epoch-Level Selection Score
# ============================================

def epoch_selection_score(
    validation_summary
):

    return float(

        validation_summary[
            "Worst_Domain_AUC"
        ]
    )

In [ ]:
# ============================================
# Section 10.7
# Train One Adaptation Epoch
# ============================================

def train_one_phase6_epoch(
    model,
    dataloader,
    optimizer,
    lambda_value,
    benign_sigma,
    malignant_sigma,
    epoch
):

    model.train()


    if hasattr(
        dataloader.batch_sampler,
        "set_epoch"
    ):

        dataloader.batch_sampler.set_epoch(
            epoch
        )


    total_loss_sum = 0.0
    ce_loss_sum = 0.0
    mmd_loss_sum = 0.0

    benign_mmd_sum = 0.0
    malignant_mmd_sum = 0.0

    correct = 0
    total = 0


    for batch in dataloader:

        images = (

            batch[
                "image"
            ]
            .to(
                DEVICE
            )
        )


        targets = (

            batch[
                "target"
            ]
            .to(
                DEVICE
            )
        )


        domains = (

            batch[
                "domain"
            ]
            .to(
                DEVICE
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits, features = model(
            images
        )


        ce_loss = F.cross_entropy(

            logits,

            targets
        )


        (
            mmd_loss,
            benign_mmd,
            malignant_mmd

        ) = class_conditional_mmd_loss(

            features=features,

            targets=targets,

            domains=domains,

            benign_sigma=benign_sigma,

            malignant_sigma=malignant_sigma
        )


        total_loss = (

            ce_loss

            +

            float(
                lambda_value
            )
            *
            mmd_loss
        )


        total_loss.backward()


        optimizer.step()


        predictions = torch.argmax(

            logits,

            dim=1
        )


        batch_size = (
            targets.shape[0]
        )


        total_loss_sum += (

            float(
                total_loss.detach().cpu()
            )

            *
            batch_size
        )


        ce_loss_sum += (

            float(
                ce_loss.detach().cpu()
            )

            *
            batch_size
        )


        mmd_loss_sum += (

            float(
                mmd_loss.detach().cpu()
            )

            *
            batch_size
        )


        benign_mmd_sum += (

            float(
                benign_mmd.detach().cpu()
            )

            *
            batch_size
        )


        malignant_mmd_sum += (

            float(
                malignant_mmd.detach().cpu()
            )

            *
            batch_size
        )


        correct += int(

            (
                predictions
                ==
                targets
            )
            .sum()
            .detach()
            .cpu()
        )


        total += (
            batch_size
        )


    return {

        "Train_Total_Loss":
            total_loss_sum
            /
            total,

        "Train_CE_Loss":
            ce_loss_sum
            /
            total,

        "Train_MMD_Loss":
            mmd_loss_sum
            /
            total,

        "Train_Benign_MMD2":
            benign_mmd_sum
            /
            total,

        "Train_Malignant_MMD2":
            malignant_mmd_sum
            /
            total,

        "Train_Accuracy":
            correct
            /
            total
    }

In [ ]:
# ============================================
# Section 10.8
# One-Epoch Training Smoke Test
# ============================================

SMOKE_SEED = 42
SMOKE_LAMBDA = 1.0


set_run_seed(
    SMOKE_SEED
)


smoke_model = (
    create_phase6_model()
)


smoke_optimizer = torch.optim.Adam(

    smoke_model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


smoke_train_metrics = (
    train_one_phase6_epoch(

        model=smoke_model,

        dataloader=phase6_train_loader,

        optimizer=smoke_optimizer,

        lambda_value=SMOKE_LAMBDA,

        benign_sigma=(
            TRAINING_MMD_BANDWIDTHS[
                "Benign"
            ]
        ),

        malignant_sigma=(
            TRAINING_MMD_BANDWIDTHS[
                "Malignant"
            ]
        ),

        epoch=0
    )
)


smoke_val_metrics = (
    evaluate_both_validation_domains(
        smoke_model
    )
)


print(
    "TRAIN:"
)

print(
    smoke_train_metrics
)


print(
    "\nVALIDATION:"
)

print(
    smoke_val_metrics
)


assert all(

    np.isfinite(
        value
    )

    for value
    in smoke_train_metrics.values()
)


assert all(

    np.isfinite(
        value
    )

    for value
    in smoke_val_metrics.values()
)


print(
    "\n✓ One-epoch Phase 6 smoke test passed."
)

In [ ]:
# ============================================
# Section 10.9
# Full Resume-Safe Epoch Checkpoint
# ============================================

def save_phase6_checkpoint(
    path,
    model,
    optimizer,
    epoch,
    best_selection_score,
    best_epoch,
    epochs_without_improvement,
    lambda_value,
    seed,
    history
):

    checkpoint = {

        "epoch":
            int(
                epoch
            ),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "best_selection_score":
            float(
                best_selection_score
            ),

        "best_epoch":
            int(
                best_epoch
            ),

        "epochs_without_improvement":
            int(
                epochs_without_improvement
            ),

        "lambda_value":
            float(
                lambda_value
            ),

        "seed":
            int(
                seed
            ),

        "history":
            history,

        # ------------------------------------
        # Random-state preservation
        # ------------------------------------

        "python_rng_state":
            random.getstate(),

        "numpy_rng_state":
            np.random.get_state(),

        "torch_rng_state":
            torch.get_rng_state(),

        "cuda_rng_state_all":
            (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else None
            )
    }


    torch.save(
        checkpoint,
        path
    )

In [ ]:
# ============================================
# Section 10.10
# Restore Interrupted Training Run
# ============================================

def restore_phase6_checkpoint(
    checkpoint_path,
    model,
    optimizer
):

    checkpoint = torch.load(

        checkpoint_path,

        map_location=DEVICE,

        weights_only=False
    )


    model.load_state_dict(

        checkpoint[
            "model_state_dict"
        ]
    )


    optimizer.load_state_dict(

        checkpoint[
            "optimizer_state_dict"
        ]
    )


    # ----------------------------------------
    # Restore random states
    # ----------------------------------------

    if (
        "python_rng_state"
        in checkpoint
    ):

        random.setstate(
            checkpoint[
                "python_rng_state"
            ]
        )


    if (
        "numpy_rng_state"
        in checkpoint
    ):

        np.random.set_state(
            checkpoint[
                "numpy_rng_state"
            ]
        )


    if (
        "torch_rng_state"
        in checkpoint
    ):

        torch.set_rng_state(
            checkpoint[
                "torch_rng_state"
            ]
        )


    if (
        torch.cuda.is_available()

        and

        checkpoint.get(
            "cuda_rng_state_all"
        )
        is not None
    ):

        torch.cuda.set_rng_state_all(

            checkpoint[
                "cuda_rng_state_all"
            ]
        )


    return checkpoint

In [ ]:
# ============================================
# Section 11.1
# Train / Resume One Lambda-Seed Run
# ============================================

def run_phase6_tuning_experiment(
    lambda_value,
    seed
):

    paths = phase6_run_paths(

        lambda_value,
        seed
    )


    # ========================================
    # Skip fully completed runs
    # ========================================

    if tuning_run_complete(
        lambda_value,
        seed
    ):

        print(
            f"✓ SKIP COMPLETE | "
            f"lambda={lambda_value} | "
            f"seed={seed}"
        )

        return


    print(
        "\n"
        +
        "=" * 80
    )

    print(
        f"PHASE 6 TUNING RUN | "
        f"lambda={lambda_value} | "
        f"seed={seed}"
    )

    print(
        "=" * 80
    )


    set_run_seed(
        seed
    )


    # ========================================
    # Fresh model + optimizer
    # ========================================

    model = (
        create_phase6_model()
    )


    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


    start_epoch = 0

    best_selection_score = (
        -np.inf
    )

    best_epoch = -1

    epochs_without_improvement = 0

    history = []


    # ========================================
    # Resume if interrupted
    # ========================================

    if paths[
        "last_checkpoint"
    ].exists():

        checkpoint = (
            restore_phase6_checkpoint(

                checkpoint_path=(
                    paths[
                        "last_checkpoint"
                    ]
                ),

                model=model,

                optimizer=optimizer
            )
        )


        checkpoint_lambda = float(

            checkpoint[
                "lambda_value"
            ]
        )


        checkpoint_seed = int(

            checkpoint[
                "seed"
            ]
        )


        assert np.isclose(

            checkpoint_lambda,

            float(
                lambda_value
            )

        ), (
            "Checkpoint lambda does not "
            "match requested lambda."
        )


        assert (
            checkpoint_seed
            ==
            int(
                seed
            )
        ), (
            "Checkpoint seed does not "
            "match requested seed."
        )


        start_epoch = (

            int(
                checkpoint[
                    "epoch"
                ]
            )

            +
            1
        )


        best_selection_score = float(

            checkpoint[
                "best_selection_score"
            ]
        )


        best_epoch = int(

            checkpoint[
                "best_epoch"
            ]
        )


        epochs_without_improvement = int(

            checkpoint[
                "epochs_without_improvement"
            ]
        )


        history = checkpoint.get(
            "history",
            []
        )


        print(
            f"↻ Resuming from epoch "
            f"{start_epoch + 1}"
        )


    # ========================================
    # Mark run IN_PROGRESS
    # ========================================

    run_manifest = {

        "Status":
            "IN_PROGRESS",

        "Lambda":
            float(
                lambda_value
            ),

        "Seed":
            int(
                seed
            ),

        "Base_Experiment":
            "M2_50F_50D",

        "Batch_Composition":
            "8FB + 8FM + 8DB + 8DM",

        "Training_MMD":
            "Biased Gaussian MMD squared",

        "Training_MMD_Class_Conditional":
            True,

        "Bandwidth_Source":
            "Training-only initial avgpool features",

        "Validation_Only_Model_Selection":
            True,

        "Test_Used":
            False
    }


    with open(

        paths[
            "manifest"
        ],

        "w"

    ) as f:

        json.dump(

            run_manifest,

            f,

            indent=4
        )


    # ========================================
    # Epoch loop
    # ========================================

    for epoch in range(

        start_epoch,

        MAX_EPOCHS
    ):

        train_metrics = (
            train_one_phase6_epoch(

                model=model,

                dataloader=(
                    phase6_train_loader
                ),

                optimizer=optimizer,

                lambda_value=(
                    lambda_value
                ),

                benign_sigma=(
                    TRAINING_MMD_BANDWIDTHS[
                        "Benign"
                    ]
                ),

                malignant_sigma=(
                    TRAINING_MMD_BANDWIDTHS[
                        "Malignant"
                    ]
                ),

                epoch=epoch
            )
        )


        val_metrics = (
            evaluate_both_validation_domains(
                model
            )
        )


        current_score = (
            epoch_selection_score(
                val_metrics
            )
        )


        row = {

            "Lambda":
                float(
                    lambda_value
                ),

            "Seed":
                int(
                    seed
                ),

            "Epoch":
                int(
                    epoch + 1
                ),

            **train_metrics,

            **val_metrics
        }


        history.append(
            row
        )


        history_df = pd.DataFrame(
            history
        )


        history_df.to_csv(

            paths[
                "history"
            ],

            index=False
        )


        # ====================================
        # Improvement check
        # ====================================

        improved = (

            current_score
            >
            best_selection_score
            +
            1e-12
        )


        if improved:

            best_selection_score = (
                current_score
            )

            best_epoch = (
                epoch
            )

            epochs_without_improvement = 0


            torch.save({

                "epoch":
                    int(
                        epoch
                    ),

                "lambda_value":
                    float(
                        lambda_value
                    ),

                "seed":
                    int(
                        seed
                    ),

                "model_state_dict":
                    model.state_dict(),

                "validation_metrics":
                    val_metrics,

                "selection_score":
                    float(
                        current_score
                    )

            }, paths[
                "best_checkpoint"
            ])


        else:

            epochs_without_improvement += 1


        # ====================================
        # Save latest state EVERY epoch
        # ====================================

        save_phase6_checkpoint(

            path=(
                paths[
                    "last_checkpoint"
                ]
            ),

            model=model,

            optimizer=optimizer,

            epoch=epoch,

            best_selection_score=(
                best_selection_score
            ),

            best_epoch=best_epoch,

            epochs_without_improvement=(
                epochs_without_improvement
            ),

            lambda_value=lambda_value,

            seed=seed,

            history=history
        )


        print(

            f"λ={lambda_value:<6} | "
            f"seed={seed} | "
            f"epoch={epoch + 1:02d} | "

            f"CE="
            f"{train_metrics['Train_CE_Loss']:.4f} | "

            f"MMD="
            f"{train_metrics['Train_MMD_Loss']:.5f} | "

            f"Fair AUC="
            f"{val_metrics['Fair_AUC']:.4f} | "

            f"Dark AUC="
            f"{val_metrics['Dark_AUC']:.4f} | "

            f"Worst="
            f"{val_metrics['Worst_Domain_AUC']:.4f} | "

            f"Recall gap="
            f"{val_metrics['Recall_Gap']:.4f}"
        )


        # ====================================
        # Early stopping
        # ====================================

        if (
            epochs_without_improvement
            >=
            EARLY_STOPPING_PATIENCE
        ):

            print(
                f"Early stopping at epoch "
                f"{epoch + 1}."
            )

            break


    # ========================================
    # Complete manifest
    # ========================================

    best_checkpoint = torch.load(

        paths[
            "best_checkpoint"
        ],

        map_location="cpu",

        weights_only=False
    )


    best_validation_metrics = (

        best_checkpoint[
            "validation_metrics"
        ]
    )


    final_manifest = {

        "Status":
            "COMPLETE",

        "Lambda":
            float(
                lambda_value
            ),

        "Seed":
            int(
                seed
            ),

        "Best_Epoch":
            int(
                best_checkpoint[
                    "epoch"
                ]
                +
                1
            ),

        "Best_Worst_Domain_AUC":
            float(
                best_validation_metrics[
                    "Worst_Domain_AUC"
                ]
            ),

        "Best_Fair_AUC":
            float(
                best_validation_metrics[
                    "Fair_AUC"
                ]
            ),

        "Best_Dark_AUC":
            float(
                best_validation_metrics[
                    "Dark_AUC"
                ]
            ),

        "Best_Mean_Domain_AUC":
            float(
                best_validation_metrics[
                    "Mean_Domain_AUC"
                ]
            ),

        "Best_AUC_Gap":
            float(
                best_validation_metrics[
                    "AUC_Gap"
                ]
            ),

        "Best_Fair_Recall":
            float(
                best_validation_metrics[
                    "Fair_Recall"
                ]
            ),

        "Best_Dark_Recall":
            float(
                best_validation_metrics[
                    "Dark_Recall"
                ]
            ),

        "Best_Recall_Gap":
            float(
                best_validation_metrics[
                    "Recall_Gap"
                ]
            ),

        "Training_MMD":
            "Biased Gaussian MMD squared",

        "Class_Conditional":
            True,

        "Test_Used":
            False
    }


    with open(

        paths[
            "manifest"
        ],

        "w"

    ) as f:

        json.dump(

            final_manifest,

            f,

            indent=4
        )


    print(
        f"\n✓ COMPLETE | "
        f"lambda={lambda_value} | "
        f"seed={seed} | "
        f"best epoch="
        f"{final_manifest['Best_Epoch']} | "
        f"worst AUC="
        f"{final_manifest['Best_Worst_Domain_AUC']:.4f}"
    )


    # Free GPU memory between runs
    del model
    del optimizer

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()

In [ ]:
# ============================================
# Section 11.2
# Freeze Lambda Tuning Grid
# ============================================

LAMBDA_GRID = [

    0.0,
    1.0,
    10.0,
    50.0,
    100.0
]


TUNING_SEEDS = [

    42,
    43,
    44
]


tuning_design = {

    "Lambda_Grid":
        LAMBDA_GRID,

    "Tuning_Seeds":
        TUNING_SEEDS,

    "Total_Runs":
        (
            len(
                LAMBDA_GRID
            )
            *
            len(
                TUNING_SEEDS
            )
        ),

    "Validation_Only":
        True,

    "Test_Used":
        False,

    "Frozen":
        True
}


with open(

    PHASE6_CONFIG_PATH
    / "Phase6_Tuning_Design.json",

    "w"

) as f:

    json.dump(

        tuning_design,

        f,

        indent=4
    )


print(
    "Lambda grid:",
    LAMBDA_GRID
)

print(
    "Tuning seeds:",
    TUNING_SEEDS
)

print(
    "Total tuning runs:",
    tuning_design[
        "Total_Runs"
    ]
)

print(
    "\n✓ Tuning design frozen."
)

In [ ]:
# ============================================
# Section 11.3A
# Missing Utility Import
# ============================================

import gc

print(
    "✓ gc imported."
)

In [ ]:
# ============================================
# Section 11.4
# Launch Resume-Safe Lambda Tuning
# ============================================

for lambda_value in LAMBDA_GRID:

    for seed in TUNING_SEEDS:

        run_phase6_tuning_experiment(

            lambda_value=lambda_value,

            seed=seed
        )


print(
    "\n✓ Phase 6 tuning loop finished."
)

In [ ]:
# ============================================
# Section 12.0
# Restore Phase 6 Tuning Configuration
# ============================================

from pathlib import Path
import json
import pandas as pd


# --------------------------------------------
# Main Phase 6 root
# --------------------------------------------

PHASE6_ROOT = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts/Phase_6_Domain_Adaptation"
)


PHASE6_TUNING_PATH = (
    PHASE6_ROOT
    / "tuning"
)


PHASE6_TABLE_PATH = (
    PHASE6_ROOT
    / "tables"
)


PHASE6_CONFIG_PATH = (
    PHASE6_ROOT
    / "config"
)


PHASE6_MANIFEST_PATH = (
    PHASE6_ROOT
    / "manifests"
)


# --------------------------------------------
# Restore frozen tuning design
# --------------------------------------------

TUNING_DESIGN_FILE = (

    PHASE6_CONFIG_PATH
    / "Phase6_Tuning_Design.json"
)


assert (
    TUNING_DESIGN_FILE.exists()
), (
    f"Tuning design file not found: "
    f"{TUNING_DESIGN_FILE}"
)


with open(
    TUNING_DESIGN_FILE,
    "r"
) as f:

    tuning_design = json.load(
        f
    )


LAMBDA_GRID = (
    tuning_design[
        "Lambda_Grid"
    ]
)


TUNING_SEEDS = (
    tuning_design[
        "Tuning_Seeds"
    ]
)


print(
    "Phase 6 root:",
    PHASE6_ROOT
)


print(
    "Lambda grid:",
    LAMBDA_GRID
)


print(
    "Tuning seeds:",
    TUNING_SEEDS
)


print(
    "Expected tuning runs:",
    len(
        LAMBDA_GRID
    )
    *
    len(
        TUNING_SEEDS
    )
)


print(
    "\n✓ Phase 6 tuning configuration restored."
)

In [ ]:
# ============================================
# Section 12.0B
# Restore Tuning Path Helpers
# ============================================

def lambda_to_name(
    lambda_value
):

    return str(
        lambda_value
    ).replace(
        ".",
        "p"
    )


def phase6_run_directory(
    lambda_value,
    seed
):

    return (

        PHASE6_TUNING_PATH

        / (
            f"lambda_{lambda_to_name(lambda_value)}"
            f"_seed_{seed}"
        )
    )


def phase6_run_paths(
    lambda_value,
    seed
):

    run_dir = (
        phase6_run_directory(
            lambda_value,
            seed
        )
    )


    return {

        "directory":
            run_dir,

        "last_checkpoint":
            run_dir
            / "last_checkpoint.pth",

        "best_checkpoint":
            run_dir
            / "best_model.pth",

        "history":
            run_dir
            / "history.csv",

        "manifest":
            run_dir
            / "run_manifest.json"
    }


print(
    "✓ Phase 6 tuning path helpers restored."
)

In [ ]:
# ============================================
# Section 12.1
# Audit Saved Phase 6 Tuning Runs
# ============================================

expected_runs = [

    (
        lambda_value,
        seed
    )

    for lambda_value in LAMBDA_GRID
    for seed in TUNING_SEEDS
]


audit_rows = []


for lambda_value, seed in expected_runs:

    paths = phase6_run_paths(
        lambda_value,
        seed
    )


    manifest_exists = (
        paths[
            "manifest"
        ]
        .exists()
    )


    best_checkpoint_exists = (
        paths[
            "best_checkpoint"
        ]
        .exists()
    )


    history_exists = (
        paths[
            "history"
        ]
        .exists()
    )


    status = None


    if manifest_exists:

        with open(
            paths[
                "manifest"
            ],
            "r"
        ) as f:

            manifest = json.load(
                f
            )


        status = manifest.get(
            "Status"
        )


    audit_rows.append({

        "Lambda":
            lambda_value,

        "Seed":
            seed,

        "Status":
            status,

        "Manifest":
            manifest_exists,

        "Best_Checkpoint":
            best_checkpoint_exists,

        "History":
            history_exists
    })


tuning_audit_df = pd.DataFrame(
    audit_rows
)


display(
    tuning_audit_df
)


complete_mask = (

    tuning_audit_df[
        "Status"
    ]
    .eq(
        "COMPLETE"
    )

    &

    tuning_audit_df[
        "Manifest"
    ]

    &

    tuning_audit_df[
        "Best_Checkpoint"
    ]

    &

    tuning_audit_df[
        "History"
    ]
)


completed_runs = int(
    complete_mask.sum()
)


print(
    "\nCompleted tuning runs:",
    completed_runs,
    "/",
    len(
        expected_runs
    )
)


assert (
    completed_runs
    ==
    len(
        expected_runs
    )
), (
    "Some tuning artifacts are incomplete."
)


print(
    "\n✓ All Phase 6 tuning runs are safely stored."
)

In [ ]:
# ============================================
# Section 12.2
# Reconstruct Tuning Results From Drive
# ============================================

tuning_result_rows = []


for lambda_value in LAMBDA_GRID:

    for seed in TUNING_SEEDS:

        paths = phase6_run_paths(
            lambda_value,
            seed
        )


        with open(
            paths[
                "manifest"
            ],
            "r"
        ) as f:

            manifest = json.load(
                f
            )


        assert (
            manifest[
                "Status"
            ]
            ==
            "COMPLETE"
        )


        tuning_result_rows.append({

            "Lambda":
                float(
                    manifest[
                        "Lambda"
                    ]
                ),

            "Seed":
                int(
                    manifest[
                        "Seed"
                    ]
                ),

            "Best_Epoch":
                int(
                    manifest[
                        "Best_Epoch"
                    ]
                ),

            "Worst_Domain_AUC":
                float(
                    manifest[
                        "Best_Worst_Domain_AUC"
                    ]
                ),

            "Fair_AUC":
                float(
                    manifest[
                        "Best_Fair_AUC"
                    ]
                ),

            "Dark_AUC":
                float(
                    manifest[
                        "Best_Dark_AUC"
                    ]
                ),

            "Mean_Domain_AUC":
                float(
                    manifest[
                        "Best_Mean_Domain_AUC"
                    ]
                ),

            "AUC_Gap":
                float(
                    manifest[
                        "Best_AUC_Gap"
                    ]
                ),

            "Fair_Recall":
                float(
                    manifest[
                        "Best_Fair_Recall"
                    ]
                ),

            "Dark_Recall":
                float(
                    manifest[
                        "Best_Dark_Recall"
                    ]
                ),

            "Recall_Gap":
                float(
                    manifest[
                        "Best_Recall_Gap"
                    ]
                )
        })


phase6_tuning_results_df = pd.DataFrame(
    tuning_result_rows
)


phase6_tuning_results_df = (
    phase6_tuning_results_df
    .sort_values(
        [
            "Lambda",
            "Seed"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    phase6_tuning_results_df
)


PHASE6_TUNING_RESULTS_FILE = (

    PHASE6_TABLE_PATH
    / "Phase6_Tuning_Run_Results.csv"
)


phase6_tuning_results_df.to_csv(

    PHASE6_TUNING_RESULTS_FILE,

    index=False
)


print(
    "\n✓ Tuning results reconstructed and saved."
)

In [ ]:
# ============================================
# Section 12.3
# Lambda-Level Validation Summary
# ============================================

phase6_lambda_summary_df = (

    phase6_tuning_results_df

    .groupby(
        "Lambda",
        as_index=False
    )

    .agg({

        "Worst_Domain_AUC":
            [
                "mean",
                "std"
            ],

        "Mean_Domain_AUC":
            [
                "mean",
                "std"
            ],

        "AUC_Gap":
            [
                "mean",
                "std"
            ],

        "Recall_Gap":
            [
                "mean",
                "std"
            ],

        "Fair_AUC":
            "mean",

        "Dark_AUC":
            "mean"
    })
)


phase6_lambda_summary_df.columns = [

    "Lambda",

    "Worst_AUC_Mean",
    "Worst_AUC_SD",

    "Mean_Domain_AUC_Mean",
    "Mean_Domain_AUC_SD",

    "AUC_Gap_Mean",
    "AUC_Gap_SD",

    "Recall_Gap_Mean",
    "Recall_Gap_SD",

    "Fair_AUC_Mean",
    "Dark_AUC_Mean"
]


display(
    phase6_lambda_summary_df
)


PHASE6_LAMBDA_SUMMARY_FILE = (

    PHASE6_TABLE_PATH
    / "Phase6_Lambda_Validation_Summary.csv"
)


phase6_lambda_summary_df.to_csv(

    PHASE6_LAMBDA_SUMMARY_FILE,

    index=False
)


print(
    "\n✓ Lambda-level summary saved."
)

In [ ]:
# ============================================
# Section 12.3A
# Restore Frozen Selection Tolerance
# ============================================

WORST_AUC_TOLERANCE = 0.002


print(
    "Worst-domain AUC tolerance:",
    WORST_AUC_TOLERANCE
)


print(
    "✓ Frozen selection tolerance restored."
)

In [ ]:
# ============================================
# Section 12.4
# Apply Frozen Lambda Selection Rule
# ============================================

selection_df = (
    phase6_lambda_summary_df
    .copy()
)


# --------------------------------------------
# Step 1
# Find maximum mean worst-domain AUC
# --------------------------------------------

best_worst_auc = float(

    selection_df[
        "Worst_AUC_Mean"
    ]
    .max()
)


print(
    "Maximum mean worst-domain AUC:",
    best_worst_auc
)


# --------------------------------------------
# Step 2
# Apply frozen practical-tie tolerance
# --------------------------------------------

eligible_df = (

    selection_df[

        selection_df[
            "Worst_AUC_Mean"
        ]

        >=

        (
            best_worst_auc
            -
            WORST_AUC_TOLERANCE
        )
    ]

    .copy()
)


print(
    "\nCandidates within frozen tolerance:"
)


display(

    eligible_df[
        [
            "Lambda",
            "Worst_AUC_Mean",
            "Recall_Gap_Mean",
            "Mean_Domain_AUC_Mean",
            "AUC_Gap_Mean"
        ]
    ]
)


# --------------------------------------------
# Step 3
# Frozen lexicographic tie-breaking
#
# 1. Lower recall gap
# 2. Higher mean-domain AUC
# 3. Smaller lambda
# --------------------------------------------

eligible_df = (

    eligible_df

    .sort_values(

        by=[
            "Recall_Gap_Mean",
            "Mean_Domain_AUC_Mean",
            "Lambda"
        ],

        ascending=[
            True,
            False,
            True
        ]
    )

    .reset_index(
        drop=True
    )
)


SELECTED_LAMBDA = float(

    eligible_df
    .iloc[0][
        "Lambda"
    ]
)


selected_row = (
    eligible_df
    .iloc[0]
)


print(
    "\nSelected lambda:",
    SELECTED_LAMBDA
)


print(
    "Selected worst-domain AUC:",
    selected_row[
        "Worst_AUC_Mean"
    ]
)


print(
    "Selected recall gap:",
    selected_row[
        "Recall_Gap_Mean"
    ]
)


print(
    "Selected mean-domain AUC:",
    selected_row[
        "Mean_Domain_AUC_Mean"
    ]
)

In [ ]:
# ============================================
# Section 12.5
# Freeze Selected Lambda
# ============================================

selection_manifest = {

    "Selected_Lambda":
        float(
            SELECTED_LAMBDA
        ),

    "Selection_Source":
        "Validation only",

    "Primary_Criterion":
        "Mean worst-domain validation AUC",

    "Worst_AUC_Tolerance":
        float(
            WORST_AUC_TOLERANCE
        ),

    "Secondary_Criterion":
        "Mean malignant recall disparity",

    "Tertiary_Criterion":
        "Mean domain AUC",

    "Final_Tie_Break":
        "Smaller lambda",

    "Test_Data_Used":
        False,

    "Selection_Frozen":
        True
}


PHASE6_SELECTED_LAMBDA_FILE = (

    PHASE6_CONFIG_PATH
    / "Phase6_Selected_Lambda.json"
)


with open(
    PHASE6_SELECTED_LAMBDA_FILE,
    "w"
) as f:

    json.dump(
        selection_manifest,
        f,
        indent=4
    )


print(
    "✓ Selected lambda frozen:",
    SELECTED_LAMBDA
)


print(
    PHASE6_SELECTED_LAMBDA_FILE
)

In [ ]:
# ============================================
# Section 13.1
# Freeze Final Phase 6 Evaluation Design
# ============================================

FINAL_SEEDS = [
    42,
    43,
    44,
    45,
    46
]


SELECTED_LAMBDA = 0.0

SENSITIVITY_LAMBDA = 1.0


phase6_final_design = {

    "Primary_Selected_Lambda":
        SELECTED_LAMBDA,

    "Sensitivity_Nonzero_Lambda":
        SENSITIVITY_LAMBDA,

    "Final_Seeds":
        FINAL_SEEDS,

    "Number_of_Seeds":
        len(
            FINAL_SEEDS
        ),

    "Primary_Interpretation":
        (
            "Lambda 0 was selected by the "
            "predefined validation-only rule."
        ),

    "Sensitivity_Interpretation":
        (
            "Lambda 1 is evaluated only as the "
            "best-performing nonzero MMD candidate."
        ),

    "Sensitivity_Is_Confirmatory_Winner":
        False,

    "Test_Used_For_Selection":
        False,

    "Test_Results_Will_Not_Change_Lambda":
        True,

    "Frozen":
        True
}


PHASE6_FINAL_DESIGN_FILE = (

    PHASE6_CONFIG_PATH
    / "Phase6_Final_Evaluation_Design.json"
)


with open(
    PHASE6_FINAL_DESIGN_FILE,
    "w"
) as f:

    json.dump(
        phase6_final_design,
        f,
        indent=4
    )


print(
    "Primary selected lambda:",
    SELECTED_LAMBDA
)

print(
    "Sensitivity lambda:",
    SENSITIVITY_LAMBDA
)

print(
    "Final seeds:",
    FINAL_SEEDS
)

print(
    "\n✓ Final Phase 6 design frozen before test evaluation."
)

In [ ]:
# ============================================
# Section 13.1B
# Restore Frozen Final Evaluation Design
# ============================================

PHASE6_FINAL_DESIGN_FILE = (
    PHASE6_CONFIG_PATH
    / "Phase6_Final_Evaluation_Design.json"
)


assert PHASE6_FINAL_DESIGN_FILE.exists()


with open(
    PHASE6_FINAL_DESIGN_FILE,
    "r"
) as f:

    phase6_final_design = json.load(f)


FINAL_SEEDS = [
    int(x)
    for x in phase6_final_design[
        "Final_Seeds"
    ]
]


SELECTED_LAMBDA = float(
    phase6_final_design[
        "Primary_Selected_Lambda"
    ]
)


SENSITIVITY_LAMBDA = float(
    phase6_final_design[
        "Sensitivity_Nonzero_Lambda"
    ]
)


print(
    "Frozen final seeds:",
    FINAL_SEEDS
)

print(
    "Selected lambda:",
    SELECTED_LAMBDA
)

print(
    "Sensitivity lambda:",
    SENSITIVITY_LAMBDA
)

print(
    "\n✓ Frozen final design restored from Drive."
)

In [ ]:
# ============================================
# Section 13.2
# Audit Existing Final-Seed Checkpoint Coverage
# ============================================

final_conditions = [
    SELECTED_LAMBDA,
    SENSITIVITY_LAMBDA
]


coverage_rows = []


for lambda_value in final_conditions:

    for seed in FINAL_SEEDS:

        paths = phase6_run_paths(
            lambda_value,
            seed
        )


        manifest_exists = (
            paths[
                "manifest"
            ].exists()
        )


        best_exists = (
            paths[
                "best_checkpoint"
            ].exists()
        )


        status = None


        if manifest_exists:

            with open(
                paths[
                    "manifest"
                ],
                "r"
            ) as f:

                manifest = json.load(
                    f
                )


            status = manifest.get(
                "Status"
            )


        coverage_rows.append({

            "Lambda":
                lambda_value,

            "Seed":
                seed,

            "Status":
                status,

            "Best_Checkpoint":
                best_exists,

            "Ready":
                (
                    status
                    ==
                    "COMPLETE"
                    and
                    best_exists
                )
        })


phase6_final_coverage_df = pd.DataFrame(
    coverage_rows
)


display(
    phase6_final_coverage_df
)


print(
    "\nReady models:",
    int(
        phase6_final_coverage_df[
            "Ready"
        ].sum()
    ),
    "/",
    len(
        phase6_final_coverage_df
    )
)


print(
    "\nModels still requiring training:"
)


display(

    phase6_final_coverage_df[

        ~phase6_final_coverage_df[
            "Ready"
        ]

    ][
        [
            "Lambda",
            "Seed"
        ]
    ]
)

In [ ]:
# ============================================
# Section 13.3
# Restore Final-Training Runtime Configuration
# ============================================

import json
import random
import gc
import numpy as np
import pandas as pd
import torch


# --------------------------------------------
# Core frozen training settings
# --------------------------------------------

GLOBAL_SEED = 42

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# --------------------------------------------
# Restore saved MMD bandwidths
# --------------------------------------------

PHASE6_BANDWIDTH_MANIFEST_FILE = (

    PHASE6_CONFIG_PATH
    / "Phase6_MMD_Bandwidth_Manifest.json"
)


assert (
    PHASE6_BANDWIDTH_MANIFEST_FILE.exists()
), (
    "Saved Phase 6 bandwidth manifest not found."
)


with open(
    PHASE6_BANDWIDTH_MANIFEST_FILE,
    "r"
) as f:

    bandwidth_manifest = json.load(
        f
    )


TRAINING_MMD_BANDWIDTHS = {

    "Benign":
        float(
            bandwidth_manifest[
                "Benign_Sigma"
            ]
        ),

    "Malignant":
        float(
            bandwidth_manifest[
                "Malignant_Sigma"
            ]
        )
}


print(
    "Device:",
    DEVICE
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Max epochs:",
    MAX_EPOCHS
)

print(
    "Patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "\nRestored bandwidths:"
)

print(
    TRAINING_MMD_BANDWIDTHS
)


print(
    "\n✓ Final-training configuration restored."
)

In [ ]:
# ============================================
# Section 13.4
# Final Training Runtime Gate
# ============================================

required_objects = [

    "phase6_train_loader",

    "fair_val_loader",
    "dark_val_loader",

    "ResNet50WithFeatures",

    "training_mmd2_torch",
    "class_conditional_mmd_loss",

    "evaluate_both_validation_domains",

    "phase6_run_paths",

    "run_phase6_tuning_experiment"
]


missing_objects = [

    name

    for name in required_objects

    if name not in globals()
]


if missing_objects:

    print(
        "Missing runtime objects:"
    )

    for name in missing_objects:

        print(
            " -",
            name
        )

else:

    print(
        "✓ Final training runtime is fully restored."
    )

In [ ]:
# ============================================
# Section 13.4A
# Fix Run-Directory Creation
# ============================================

def lambda_to_name(
    lambda_value
):

    return str(
        lambda_value
    ).replace(
        ".",
        "p"
    )


def phase6_run_directory(
    lambda_value,
    seed
):

    return (

        PHASE6_TUNING_PATH

        / (
            f"lambda_{lambda_to_name(lambda_value)}"
            f"_seed_{seed}"
        )
    )


def phase6_run_paths(
    lambda_value,
    seed
):

    run_dir = (
        phase6_run_directory(
            lambda_value,
            seed
        )
    )


    # ----------------------------------------
    # CRITICAL:
    # create directory for new seed/run
    # ----------------------------------------

    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    return {

        "directory":
            run_dir,

        "last_checkpoint":
            run_dir
            / "last_checkpoint.pth",

        "best_checkpoint":
            run_dir
            / "best_model.pth",

        "history":
            run_dir
            / "history.csv",

        "manifest":
            run_dir
            / "run_manifest.json"
    }


print(
    "✓ Phase 6 run-directory helper fixed."
)

In [ ]:
# ============================================
# Section 13.4B
# Verify Missing Final Run Directories
# ============================================

for lambda_value, seed in [

    (0.0, 45),
    (0.0, 46),
    (1.0, 45),
    (1.0, 46)

]:

    paths = phase6_run_paths(
        lambda_value,
        seed
    )


    print(
        f"lambda={lambda_value}, "
        f"seed={seed} -> "
        f"{paths['directory'].exists()}"
    )

In [ ]:
# ============================================
# Section 13.5
# Train Only Missing Final Models
# ============================================

FINAL_TRAINING_JOBS = [

    (
        0.0,
        45
    ),

    (
        0.0,
        46
    ),

    (
        1.0,
        45
    ),

    (
        1.0,
        46
    )
]


print(
    "Final models requiring training:",
    len(
        FINAL_TRAINING_JOBS
    )
)


for lambda_value, seed in FINAL_TRAINING_JOBS:

    run_phase6_tuning_experiment(

        lambda_value=lambda_value,

        seed=seed
    )


print(
    "\n✓ Missing Phase 6 final models completed."
)

In [ ]:
# ============================================
# Section 13.6
# Train Remaining Lambda=1 Final Models
# ============================================

REMAINING_FINAL_JOBS = [

    (
        1.0,
        45
    ),

    (
        1.0,
        46
    )
]


print(
    "Remaining final models:",
    len(
        REMAINING_FINAL_JOBS
    )
)


for lambda_value, seed in (
    REMAINING_FINAL_JOBS
):

    print(
        "\nStarting:",
        f"lambda={lambda_value},",
        f"seed={seed}"
    )


    run_phase6_tuning_experiment(

        lambda_value=lambda_value,

        seed=seed
    )


print(
    "\n✓ Remaining Phase 6 final models completed."
)

In [ ]:
# ============================================
# Section 14.1
# Build Fixed Fair and Dark Test Loaders
# ============================================

fair_test_dataset = Phase6Dataset(

    dataframe=fair_test_df,

    transform=eval_transform
)


dark_test_dataset = Phase6Dataset(

    dataframe=dark_test_df,

    transform=eval_transform
)


fair_test_loader = DataLoader(

    fair_test_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


dark_test_loader = DataLoader(

    dark_test_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Fair test images:",
    len(
        fair_test_dataset
    )
)


print(
    "Dark test images:",
    len(
        dark_test_dataset
    )
)


assert (
    len(
        fair_test_dataset
    )
    ==
    487
)


assert (
    len(
        dark_test_dataset
    )
    ==
    162
)


print(
    "\n✓ Fixed test loaders ready."
)

In [ ]:
# ============================================
# Section 14.2
# Final Test Evaluation Helper
# ============================================

def evaluate_phase6_test_model(
    model,
    dataloader
):

    model.eval()


    all_labels = []
    all_probabilities = []
    all_predictions = []
    all_md5 = []


    with torch.no_grad():

        for batch in dataloader:

            images = (
                batch[
                    "image"
                ]
                .to(
                    DEVICE
                )
            )


            targets = (
                batch[
                    "target"
                ]
                .to(
                    DEVICE
                )
            )


            logits, _ = model(
                images
            )


            probabilities = torch.softmax(
                logits,
                dim=1

            )[:, 1]


            predictions = torch.argmax(
                logits,
                dim=1
            )


            all_labels.extend(
                targets
                .cpu()
                .numpy()
                .tolist()
            )


            all_probabilities.extend(
                probabilities
                .cpu()
                .numpy()
                .tolist()
            )


            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
                .tolist()
            )


            all_md5.extend(
                list(
                    batch[
                        "md5hash"
                    ]
                )
            )


    metrics = compute_phase6_metrics(

        labels=all_labels,

        probabilities=all_probabilities,

        predictions=all_predictions
    )


    prediction_df = pd.DataFrame({

        "md5hash":
            all_md5,

        "target":
            all_labels,

        "probability":
            all_probabilities,

        "prediction":
            all_predictions
    })


    return (
        metrics,
        prediction_df
    )

In [ ]:
# ============================================
# Section 14.3
# Load Frozen Best Checkpoint
# ============================================

def load_phase6_best_model(
    lambda_value,
    seed
):

    paths = phase6_run_paths(
        lambda_value,
        seed
    )


    assert (
        paths[
            "best_checkpoint"
        ].exists()
    ), (
        f"Missing checkpoint: "
        f"lambda={lambda_value}, seed={seed}"
    )


    checkpoint = torch.load(

        paths[
            "best_checkpoint"
        ],

        map_location=DEVICE,

        weights_only=False
    )


    model = create_phase6_model()


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    model.eval()


    return (
        model,
        checkpoint
    )

In [ ]:
# ============================================
# Section 14.4
# Locked Final Fair / Dark Test Evaluation
# ============================================

FINAL_CONDITIONS = [
    0.0,
    1.0
]


final_test_rows = []
all_prediction_frames = []


for lambda_value in FINAL_CONDITIONS:

    for seed in FINAL_SEEDS:

        print(
            f"Evaluating "
            f"lambda={lambda_value}, "
            f"seed={seed}"
        )


        model, checkpoint = (
            load_phase6_best_model(
                lambda_value,
                seed
            )
        )


        (
            fair_metrics,
            fair_predictions

        ) = evaluate_phase6_test_model(

            model,
            fair_test_loader
        )


        (
            dark_metrics,
            dark_predictions

        ) = evaluate_phase6_test_model(

            model,
            dark_test_loader
        )


        row = {

            "Lambda":
                float(
                    lambda_value
                ),

            "Seed":
                int(
                    seed
                ),

            "Best_Epoch":
                int(
                    checkpoint[
                        "epoch"
                    ]
                    +
                    1
                ),

            "Fair_AUC":
                fair_metrics[
                    "AUC"
                ],

            "Dark_AUC":
                dark_metrics[
                    "AUC"
                ],

            "Fair_Accuracy":
                fair_metrics[
                    "Accuracy"
                ],

            "Dark_Accuracy":
                dark_metrics[
                    "Accuracy"
                ],

            "Fair_Precision":
                fair_metrics[
                    "Precision"
                ],

            "Dark_Precision":
                dark_metrics[
                    "Precision"
                ],

            "Fair_Recall":
                fair_metrics[
                    "Recall"
                ],

            "Dark_Recall":
                dark_metrics[
                    "Recall"
                ],

            "Fair_F1":
                fair_metrics[
                    "F1"
                ],

            "Dark_F1":
                dark_metrics[
                    "F1"
                ],

            "Fair_FNR":
                fair_metrics[
                    "FNR"
                ],

            "Dark_FNR":
                dark_metrics[
                    "FNR"
                ]
        }


        row[
            "AUC_Gap"
        ] = abs(

            row[
                "Fair_AUC"
            ]

            -

            row[
                "Dark_AUC"
            ]
        )


        row[
            "Recall_Gap"
        ] = abs(

            row[
                "Fair_Recall"
            ]

            -

            row[
                "Dark_Recall"
            ]
        )


        row[
            "FNR_Gap"
        ] = abs(

            row[
                "Fair_FNR"
            ]

            -

            row[
                "Dark_FNR"
            ]
        )


        row[
            "Worst_Domain_AUC"
        ] = min(

            row[
                "Fair_AUC"
            ],

            row[
                "Dark_AUC"
            ]
        )


        row[
            "Mean_Domain_AUC"
        ] = (

            row[
                "Fair_AUC"
            ]

            +

            row[
                "Dark_AUC"
            ]

        ) / 2.0


        final_test_rows.append(
            row
        )


        fair_predictions[
            "Lambda"
        ] = lambda_value

        fair_predictions[
            "Seed"
        ] = seed

        fair_predictions[
            "Domain"
        ] = "Fair"


        dark_predictions[
            "Lambda"
        ] = lambda_value

        dark_predictions[
            "Seed"
        ] = seed

        dark_predictions[
            "Domain"
        ] = "Dark"


        all_prediction_frames.extend([
            fair_predictions,
            dark_predictions
        ])


        del model

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


phase6_final_test_df = pd.DataFrame(
    final_test_rows
)


phase6_final_predictions_df = pd.concat(

    all_prediction_frames,

    ignore_index=True
)


display(
    phase6_final_test_df
)

In [ ]:
# ============================================
# Section 14.5
# Save Locked Final Test Results
# ============================================

PHASE6_FINAL_TEST_FILE = (

    PHASE6_TABLE_PATH
    / "Phase6_Final_Test_Results.csv"
)


PHASE6_FINAL_PREDICTION_FILE = (

    PHASE6_PREDICTION_PATH
    / "Phase6_Final_Test_Predictions.csv"
)


phase6_final_test_df.to_csv(

    PHASE6_FINAL_TEST_FILE,

    index=False
)


phase6_final_predictions_df.to_csv(

    PHASE6_FINAL_PREDICTION_FILE,

    index=False
)


print(
    "✓ Final test results saved:"
)

print(
    PHASE6_FINAL_TEST_FILE
)


print(
    "\n✓ Final prediction-level results saved:"
)

print(
    PHASE6_FINAL_PREDICTION_FILE
)

In [ ]:
# ============================================
# Section 15.1
# Paired Final Test Descriptive Summary
# ============================================

SUMMARY_METRICS = [

    "Fair_AUC",
    "Dark_AUC",

    "Worst_Domain_AUC",
    "Mean_Domain_AUC",

    "AUC_Gap",
    "Recall_Gap",
    "FNR_Gap",

    "Fair_Recall",
    "Dark_Recall",

    "Fair_F1",
    "Dark_F1"
]


summary_rows = []


for lambda_value in [
    0.0,
    1.0
]:

    subset = (

        phase6_final_test_df[

            phase6_final_test_df[
                "Lambda"
            ]
            ==
            lambda_value
        ]

        .sort_values(
            "Seed"
        )
    )


    row = {

        "Lambda":
            lambda_value,

        "N_Seeds":
            len(
                subset
            )
    }


    for metric in SUMMARY_METRICS:

        values = (
            subset[
                metric
            ]
            .to_numpy(
                dtype=float
            )
        )


        row[
            f"{metric}_Mean"
        ] = float(
            np.mean(
                values
            )
        )


        row[
            f"{metric}_SD"
        ] = float(
            np.std(
                values,
                ddof=1
            )
        )


    summary_rows.append(
        row
    )


phase6_final_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


display(
    phase6_final_summary_df.T
)

In [ ]:
# ============================================
# Section 15.2
# Seed-Matched Lambda=1 Minus Lambda=0 Differences
# ============================================

lambda0_df = (

    phase6_final_test_df[

        phase6_final_test_df[
            "Lambda"
        ]
        ==
        0.0
    ]

    .sort_values(
        "Seed"
    )

    .reset_index(
        drop=True
    )
)


lambda1_df = (

    phase6_final_test_df[

        phase6_final_test_df[
            "Lambda"
        ]
        ==
        1.0
    ]

    .sort_values(
        "Seed"
    )

    .reset_index(
        drop=True
    )
)


assert (
    lambda0_df[
        "Seed"
    ]
    .tolist()
    ==
    lambda1_df[
        "Seed"
    ]
    .tolist()
)


paired_rows = []


for seed_index, seed in enumerate(
    lambda0_df[
        "Seed"
    ]
):

    row = {

        "Seed":
            int(
                seed
            )
    }


    for metric in SUMMARY_METRICS:

        value_0 = float(

            lambda0_df.loc[
                seed_index,
                metric
            ]
        )


        value_1 = float(

            lambda1_df.loc[
                seed_index,
                metric
            ]
        )


        row[
            f"{metric}_Lambda0"
        ] = value_0


        row[
            f"{metric}_Lambda1"
        ] = value_1


        row[
            f"{metric}_Delta_1_minus_0"
        ] = (
            value_1
            -
            value_0
        )


    paired_rows.append(
        row
    )


phase6_paired_delta_df = (
    pd.DataFrame(
        paired_rows
    )
)


display(
    phase6_paired_delta_df
)

In [ ]:
# ============================================
# Section 15.3
# Exact Paired Wilcoxon Tests
# Lambda=1 Versus Lambda=0
# ============================================

from scipy.stats import wilcoxon


TEST_METRICS = [

    "Worst_Domain_AUC",
    "Mean_Domain_AUC",

    "Fair_AUC",
    "Dark_AUC",

    "AUC_Gap",
    "Recall_Gap",
    "FNR_Gap"
]


wilcoxon_rows = []


for metric in TEST_METRICS:

    values_0 = (

        lambda0_df[
            metric
        ]
        .to_numpy(
            dtype=float
        )
    )


    values_1 = (

        lambda1_df[
            metric
        ]
        .to_numpy(
            dtype=float
        )
    )


    differences = (
        values_1
        -
        values_0
    )


    if np.allclose(
        differences,
        0
    ):

        statistic = 0.0
        p_value = 1.0

    else:

        result = wilcoxon(

            values_1,
            values_0,

            alternative="two-sided",

            method="exact"
        )


        statistic = float(
            result.statistic
        )


        p_value = float(
            result.pvalue
        )


    wilcoxon_rows.append({

        "Metric":
            metric,

        "Lambda0_Mean":
            float(
                np.mean(
                    values_0
                )
            ),

        "Lambda1_Mean":
            float(
                np.mean(
                    values_1
                )
            ),

        "Mean_Delta_1_minus_0":
            float(
                np.mean(
                    differences
                )
            ),

        "Median_Delta_1_minus_0":
            float(
                np.median(
                    differences
                )
            ),

        "Wilcoxon_W":
            statistic,

        "Exact_P":
            p_value
    })


phase6_wilcoxon_df = pd.DataFrame(
    wilcoxon_rows
)


display(
    phase6_wilcoxon_df
)

In [ ]:
# ============================================
# Section 15.4
# Holm Correction Across Final Phase 6 Tests
# ============================================

from statsmodels.stats.multitest import (
    multipletests
)


raw_p_values = (

    phase6_wilcoxon_df[
        "Exact_P"
    ]
    .to_numpy(
        dtype=float
    )
)


reject, corrected_p, _, _ = (
    multipletests(

        raw_p_values,

        alpha=0.05,

        method="holm"
    )
)


phase6_wilcoxon_df[
    "Holm_P"
] = corrected_p


phase6_wilcoxon_df[
    "Holm_Supported"
] = reject


display(
    phase6_wilcoxon_df
)

In [ ]:
# ============================================
# Section 15.5
# Save Final Phase 6 Statistical Tables
# ============================================

PHASE6_FINAL_SUMMARY_FILE = (

    PHASE6_TABLE_PATH
    / "Phase6_Final_Test_Descriptive_Summary.csv"
)


PHASE6_PAIRED_DELTA_FILE = (

    PHASE6_TABLE_PATH
    / "Phase6_Final_Paired_Differences.csv"
)


PHASE6_WILCOXON_FILE = (

    PHASE6_TABLE_PATH
    / "Phase6_Final_Paired_Wilcoxon_Holm.csv"
)


phase6_final_summary_df.to_csv(

    PHASE6_FINAL_SUMMARY_FILE,

    index=False
)


phase6_paired_delta_df.to_csv(

    PHASE6_PAIRED_DELTA_FILE,

    index=False
)


phase6_wilcoxon_df.to_csv(

    PHASE6_WILCOXON_FILE,

    index=False
)


print(
    "✓ Final Phase 6 statistical tables saved."
)

print(
    PHASE6_FINAL_SUMMARY_FILE
)

print(
    PHASE6_PAIRED_DELTA_FILE
)

print(
    PHASE6_WILCOXON_FILE
)

In [ ]:
# ============================================
# Section 16.1
# Create Final Phase 6 Completion Manifest
# ============================================

supported_results = (

    phase6_wilcoxon_df[
        phase6_wilcoxon_df[
            "Holm_Supported"
        ]
        ==
        True
    ][
        "Metric"
    ]
    .tolist()
)


phase6_completion_manifest = {

    "Phase":
        "Phase 6",

    "Title":
        "Domain Adaptation / Mitigation",

    "Base_Experiment":
        "M2_50F_50D",

    "Adaptation_Method":
        "Class-Conditional MMD Regularization",

    "Feature_Layer":
        "ResNet50 avgpool",

    "Training_MMD_Estimator":
        "Biased non-negative empirical Gaussian MMD squared",

    "Lambda_Grid":
        [
            0.0,
            1.0,
            10.0,
            50.0,
            100.0
        ],

    "Tuning_Seeds":
        [
            42,
            43,
            44
        ],

    "Final_Seeds":
        [
            42,
            43,
            44,
            45,
            46
        ],

    "Selected_Lambda":
        0.0,

    "Sensitivity_Lambda":
        1.0,

    "Lambda_Selection_Source":
        "Validation only",

    "Test_Data_Used_For_Selection":
        False,

    "Final_Number_Models":
        10,

    "Lambda0_Worst_Domain_AUC_Mean":
        0.827458,

    "Lambda1_Worst_Domain_AUC_Mean":
        0.821295,

    "Lambda0_Mean_Domain_AUC_Mean":
        0.851933,

    "Lambda1_Mean_Domain_AUC_Mean":
        0.847092,

    "Lambda0_AUC_Gap_Mean":
        0.048949,

    "Lambda1_AUC_Gap_Mean":
        0.051594,

    "Lambda0_Recall_Gap_Mean":
        0.028820,

    "Lambda1_Recall_Gap_Mean":
        0.013328,

    "Recall_Gap_Relative_Reduction_Percent":
        (
            (
                0.028820
                -
                0.013328
            )
            /
            0.028820
            *
            100.0
        ),

    "Number_Statistical_Tests":
        int(
            len(
                phase6_wilcoxon_df
            )
        ),

    "Multiple_Testing_Correction":
        "Holm",

    "Holm_Supported_Outcomes":
        supported_results,

    "Number_Holm_Supported_Outcomes":
        len(
            supported_results
        ),

    "Primary_Result":
        (
            "The predefined validation-selection procedure "
            "selected lambda=0.0. On held-out test data, "
            "lambda=1.0 did not provide statistically "
            "supported improvement in worst-domain AUC, "
            "mean-domain AUC, Fair AUC, Dark AUC, AUC gap, "
            "recall gap, or FNR gap."
        ),

    "Sensitivity_Result":
        (
            "Lambda=1.0 descriptively reduced the mean "
            "Fair-Dark malignant recall/FNR gap from "
            "0.028820 to 0.013328, approximately a 53.8% "
            "relative reduction, but this effect was not "
            "statistically supported and was accompanied "
            "by modest decreases in discrimination."
        ),

    "Final_Interpretation":
        (
            "Class-conditional MMD regularization did not "
            "demonstrate statistically supported improvement "
            "over the no-MMD control. The selected Phase 6 "
            "configuration therefore remains lambda=0.0."
        ),

    "Status":
        "COMPLETE",

    "Frozen":
        True
}


PHASE6_COMPLETION_FILE = (

    PHASE6_MANIFEST_PATH
    / "Phase6_Completion_Manifest.json"
)


with open(
    PHASE6_COMPLETION_FILE,
    "w"
) as f:

    json.dump(
        phase6_completion_manifest,
        f,
        indent=4
    )


print(
    "✓ Phase 6 completion manifest saved:"
)

print(
    PHASE6_COMPLETION_FILE
)

print(
    "\nStatus:",
    phase6_completion_manifest[
        "Status"
    ]
)

print(
    "Frozen:",
    phase6_completion_manifest[
        "Frozen"
    ]
)

print(
    "Holm-supported outcomes:",
    phase6_completion_manifest[
        "Number_Holm_Supported_Outcomes"
    ]
)

In [ ]:
# ============================================
# Section 16.2
# Final Phase 6 Hard Completion Gate
# ============================================

assert (
    PHASE6_COMPLETION_FILE.exists()
)


with open(
    PHASE6_COMPLETION_FILE,
    "r"
) as f:

    final_phase6_manifest = json.load(
        f
    )


assert (
    final_phase6_manifest[
        "Status"
    ]
    ==
    "COMPLETE"
)


assert (
    final_phase6_manifest[
        "Frozen"
    ]
    is True
)


assert (
    final_phase6_manifest[
        "Selected_Lambda"
    ]
    ==
    0.0
)


assert (
    final_phase6_manifest[
        "Test_Data_Used_For_Selection"
    ]
    is False
)


assert (
    len(
        phase6_final_test_df
    )
    ==
    10
)


assert (
    phase6_final_test_df[
        "Seed"
    ]
    .nunique()
    ==
    5
)


assert (
    set(
        phase6_final_test_df[
            "Lambda"
        ]
    )
    ==
    {
        0.0,
        1.0
    }
)


print(
    "✓ Phase 6 status: COMPLETE"
)

print(
    "✓ Phase 6 frozen: TRUE"
)

print(
    "✓ Lambda selection used validation only."
)

print(
    "✓ Final test evaluation contains 10 models."
)

print(
    "✓ No further Phase 6 model selection is permitted."
)